## Environments

In [ ]:
!pip install -q librosa easydict packaging \
                hear21passt timm torchcodec

In [ ]:
# !pip uninstall -y mamba-ssm causal-conv1d

# !pip install causal-conv1d mamba-ssm --no-build-isolation

# !pip install flash-attn

In [1]:
from abc import ABC, abstractmethod
from collections import Counter, OrderedDict, defaultdict
from datetime import datetime
from os.path import exists, join
from pathlib import Path
from typing import Any, Callable, Dict, List, Literal, Optional, Tuple, Union

import copy
import gc
import json
import shutil
import logging
import math
import nltk
import numpy as np
import os
import pandas as pd
import pickle
import plotly.express as px
import random
import re
import seaborn as sns
import tarfile
import time
import warnings
import yaml
import zipfile
import concurrent


import librosa
import matplotlib.pyplot as plt
import torch
import torchaudio
import wandb
import multiprocessing as mp
from dotenv import load_dotenv
from easydict import EasyDict

# from google.colab import userdata
from huggingface_hub import login as hf_login, snapshot_download
from IPython.display import Audio, display
from nltk.corpus import wordnet
from torch import Tensor
from torch.nn.functional import dropout, linear, pad, softmax
from torch.nn.init import constant_
from torch.nn.modules.linear import Linear
from torch.nn.modules.module import Module
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from transformers import (
    AutoConfig,
    AutoModel,
    AutoProcessor,
    AutoTokenizer,
    ClapConfig,
    ClapFeatureExtractor,
    ClapModel,
    ClapProcessor,
    EarlyStoppingCallback,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    pipeline,
)

import torch.nn as nn
import torch.nn.functional as F

try:
    from torch.overrides import has_torch_function, handle_torch_function
except:
    from torch._overrides import has_torch_function, handle_torch_function


load_dotenv()


HF_TOKEN = os.getenv("HF_TOKEN")
WANDB_API_KEY = os.getenv("WANDB_API_KEY")
# HF_TOKEN = userdata.get('HF_TOKEN')
PROJECT_NAME = ""
WANDB_PROJECT_NAME = "[DCASE2026] Task6"
# WANDB_API_KEY = userdata.get('WANDB_API_KEY')
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
wandb.login(key=WANDB_API_KEY)
hf_login(HF_TOKEN)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/minhld/.netrc
wandb: Currently logged in as: octoopt to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
import logging

logger = logging.getLogger(__name__)
logging.basicConfig(
    format="%(asctime)s.%(msecs)03d:%(levelname)s:%(name)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
)

In [ ]:
# nltk.download('wordnet', quiet=True)
# nltk.download('omw-1.4', quiet=True)

In [3]:
# LOCAL_DIR = Path("/content/drive/MyDrive/Dataset/DCASE2026").resolve()
# DATA_DIR = LOCAL_DIR / "CLOTHO-MOMENT"

LOCAL_DIR = Path("./").resolve()
DATA_DIR = LOCAL_DIR / "data"

os.listdir(DATA_DIR)

['clotho_moment_train_release.jsonl',
 'clotho_moment_valid_test.jsonl',
 'clotho-moment',
 'clotho_moment_valid_train.jsonl',
 'preprocessed',
 'clotho-moment.zip',
 'clotho_moment_test_release.jsonl',
 'clotho_moment_val_release.jsonl',
 'clotho_moment_valid_val.jsonl']

In [4]:
TRAIN_DIR = DATA_DIR / "train"
VAL_DIR = DATA_DIR / "valid"
TEST_DIR = DATA_DIR / "test"
PREPROCESSED_DIR = DATA_DIR / "preprocessed"
FEATURES_DIR = DATA_DIR / "features"


# print(f"Examples from train dir: {os.listdir(TRAIN_DIR)[:5]}")
# print(f"Examples from validation dir: {os.listdir(VAL_DIR)[:5]}")
# print(f"Examples from test dir: {os.listdir(TEST_DIR)[:5]}")
# print(f"Examples from pre-processed dir: {os.listdir(PREPROCESSED_DIR)[:5]}")

## Experiments


In [5]:
def get_run_name(prefix, lr: float = 2e-5, batch_size: int = 256):
    now = datetime.now().strftime("%m%d-%H%M")
    return f"{prefix}_lr{lr}_bs{batch_size}_{now}"


get_run_name(prefix="test-run-name")

'test-run-name_lr2e-05_bs256_0509-1237'

In [6]:
def clear_gpu_cache():
    print(f"[Before] Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"[Before] Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

    # Clear GPU cache
    torch.cuda.empty_cache()
    # Run garbage collector
    gc.collect()

    # Verify memory is cleared
    print(f"[After] Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"[After] Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


# Verify memory is cleared
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

Allocated: 0.00 GB
Reserved:  0.00 GB


In [7]:
def set_seed(seed, use_cuda=True):
    """Sets the random seed.

    Args:
        seed (int): Seed.
        use_cuda (bool): Use cuda.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if use_cuda:
        torch.cuda.manual_seed_all(seed)

### Basic Utils

In [8]:
class WandbLogger:
    def __init__(
        self, project_name: str, run_name: str, config: dict = None, entity: str = None
    ):
        """
        Initializes the W&B run.
        """
        self.run = wandb.init(
            project=project_name,
            name=run_name,
            config=config,
            entity=entity,
            reinit=True,
        )
        self.best_accuracy = 0.0

    def log_metrics(self, metrics, step, prefix="eval"):
        """
        Logs a dictionary of metrics.

        Args:
            metrics (dict): The dictionary from your eval function.
            step (int): Current global step or epoch.
            prefix (str): Dashboard grouping (e.g., 'train' or 'eval').
        """
        # Format keys: {'loss': 0.5} -> {'eval/loss': 0.5}
        log_dict = {f"{prefix}/{k}": v for k, v in metrics.items()}

        # Log to W&B
        self.run.log(log_dict, step=step)

        # Optional: Track Best Metric Logic
        if "accuracy" in metrics:
            if metrics["accuracy"] > self.best_accuracy:
                self.best_accuracy = metrics["accuracy"]
                self.run.summary["best_accuracy"] = self.best_accuracy
                print(f"New best accuracy: {self.best_accuracy:.4f}")

    def log_artifact(self, model_path, name="model-checkpoint", aliases=["latest"]):
        artifact = wandb.Artifact(name, type="model")
        artifact.add_file(model_path)
        # Log with aliases like 'best' or 'production'
        self.run.log_artifact(artifact, aliases=aliases)

    def finish(self):
        """Closes the W&B run"""
        self.run.finish()

In [9]:
def write_log(opt, epoch_i, loss_meters, metrics=None, mode="train", **kwargs):

    wandb_logger: WandbLogger = kwargs.get("wandb_logger", None)
    if wandb_logger is not None:
        wandb_logger.log_metrics(loss_meters, epoch_i)
    if mode == "train":
        to_write = opt.train_log_txt_formatter.format(
            time_str=time.strftime("%Y_%m_%d_%H_%M_%S"),
            epoch=epoch_i + 1,
            loss_str=" ".join(
                ["{} {:.4f}".format(k, v.avg) for k, v in loss_meters.items()]
            ),
        )
        filename = opt.train_log_filepath
    else:
        to_write = opt.eval_log_txt_formatter.format(
            time_str=time.strftime("%Y_%m_%d_%H_%M_%S"),
            epoch=epoch_i,
            loss_str=" ".join(
                ["{} {:.4f}".format(k, v.avg) for k, v in loss_meters.items()]
            ),
            eval_metrics_str=json.dumps(metrics),
        )
        filename = opt.eval_log_filepath

    with open(filename, "a") as f:
        f.write(to_write)


def save_checkpoint(model, optimizer, lr_scheduler, epoch_i, opt, **kwargs):
    wandb_logger = kwargs.get("wandb_logger", None)
    wandb_artifact_version = kwargs.get("wandb_artifact_version", [])
    checkpoint = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "lr_scheduler": lr_scheduler.state_dict(),
        "epoch": epoch_i,
        "opt": opt,
    }
    torch.save(checkpoint, opt.ckpt_filepath)
    if wandb_logger is not None:
        wandb_logger.log_artifact(
            opt.ckpt_filepath,
            aliases=["latest"]
            if wandb_artifact_version is None
            else [wandb_artifact_version],
        )


def rename_latest_to_best(latest_file_paths):
    best_file_paths = [e.replace("latest", "best") for e in latest_file_paths]
    for src, tgt in zip(latest_file_paths, best_file_paths):
        os.renames(src, tgt)


def load_pickle(filename):
    with open(filename, "rb") as f:
        return pickle.load(f)


def save_pickle(data, filename):
    with open(filename, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)


def load_json(filename):
    with open(filename, "r") as f:
        return json.load(f)


def save_json(data, filename, save_pretty=False, sort_keys=False):
    with open(filename, "w") as f:
        if save_pretty:
            f.write(json.dumps(data, indent=4, sort_keys=sort_keys))
        else:
            json.dump(data, f)


def load_jsonl(filename):
    with open(filename, "r") as f:
        return [json.loads(l.strip("\n")) for l in f.readlines()]


def save_jsonl(data, filename):
    """data is a list"""
    with open(filename, "w") as f:
        f.write("\n".join([json.dumps(e) for e in data]))


def save_lines(list_of_str, filepath):
    with open(filepath, "w") as f:
        f.write("\n".join(list_of_str))


def read_lines(filepath):
    with open(filepath, "r") as f:
        return [e.strip("\n") for e in f.readlines()]


def read_yaml(file_path: Union[str, Path]) -> Dict[str, Any]:
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"YAML file not found: {file_path}")

    with open(file_path, "r", encoding="utf-8") as f:
        try:
            data = yaml.safe_load(f)
            return data if data is not None else {}
        except yaml.YAMLError as e:
            raise yaml.YAMLError(f"Error parsing YAML file {file_path}: {e}")


def mkdirp(p):
    if not os.path.exists(p):
        os.makedirs(p)


def flat_list_of_lists(l):
    """flatten a list of lists [[1,2], [3,4]] to [1,2,3,4]"""
    return [item for sublist in l for item in sublist]


def convert_to_seconds(hms_time):
    """convert '00:01:12' to 72 seconds.
    :hms_time (str): time in comma separated string, e.g. '00:01:12'
    :return (int): time in seconds, e.g. 72
    """
    times = [float(t) for t in hms_time.split(":")]
    return times[0] * 3600 + times[1] * 60 + times[2]


def get_video_name_from_url(url):
    return url.split("/")[-1][:-4]


def merge_dicts(list_dicts):
    merged_dict = list_dicts[0].copy()
    for i in range(1, len(list_dicts)):
        merged_dict.update(list_dicts[i])
    return merged_dict


def l2_normalize_np_array(np_array, eps=1e-5):
    """np_array: np.ndarray, (*, D), where the last dim will be normalized"""
    return np_array / (np.linalg.norm(np_array, axis=-1, keepdims=True) + eps)


def make_zipfile(
    src_dir,
    save_path,
    enclosing_dir="",
    exclude_dirs=None,
    exclude_extensions=None,
    exclude_dirs_substring=None,
):
    """make a zip file of root_dir, save it to save_path.
    exclude_paths will be excluded if it is a subdir of root_dir.
    An enclosing_dir is added is specified.
    """
    abs_src = os.path.abspath(src_dir)
    with zipfile.ZipFile(save_path, "w") as zf:
        for dirname, subdirs, files in os.walk(src_dir):
            if exclude_dirs is not None:
                for e_p in exclude_dirs:
                    if e_p in subdirs:
                        subdirs.remove(e_p)
            if exclude_dirs_substring is not None:
                to_rm = []
                for d in subdirs:
                    if exclude_dirs_substring in d:
                        to_rm.append(d)
                for e in to_rm:
                    subdirs.remove(e)
            arcname = os.path.join(enclosing_dir, dirname[len(abs_src) + 1 :])
            zf.write(dirname, arcname)
            for filename in files:
                if exclude_extensions is not None:
                    if os.path.splitext(filename)[1] in exclude_extensions:
                        continue  # do not zip it
                absname = os.path.join(dirname, filename)
                arcname = os.path.join(enclosing_dir, absname[len(abs_src) + 1 :])
                zf.write(absname, arcname)


class AverageMeter(object):
    """Computes and stores the average and current/max/min value"""

    def __init__(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
        self.max = -1e10
        self.min = 1e10
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
        self.max = -1e10
        self.min = 1e10

    def update(self, val, n=1):
        self.max = max(val, self.max)
        self.min = min(val, self.min)
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def dissect_by_lengths(np_array, lengths, dim=0, assert_equal=True):
    """Dissect an array (N, D) into a list a sub-array,
    np_array.shape[0] == sum(lengths), Output is a list of nd arrays, singlton dimention is kept"""
    if assert_equal:
        assert len(np_array) == sum(lengths)
    length_indices = [
        0,
    ]
    for i in range(len(lengths)):
        length_indices.append(length_indices[i] + lengths[i])
    if dim == 0:
        array_list = [
            np_array[length_indices[i] : length_indices[i + 1]]
            for i in range(len(lengths))
        ]
    elif dim == 1:
        array_list = [
            np_array[:, length_indices[i] : length_indices[i + 1]]
            for i in range(len(lengths))
        ]
    elif dim == 2:
        array_list = [
            np_array[:, :, length_indices[i] : length_indices[i + 1]]
            for i in range(len(lengths))
        ]
    else:
        raise NotImplementedError
    return array_list


def get_ratio_from_counter(counter_obj, threshold=200):
    keys = counter_obj.keys()
    values = counter_obj.values()
    filtered_values = [counter_obj[k] for k in keys if k > threshold]
    return float(sum(filtered_values)) / sum(values)


def get_counter_dist(counter_object, sort_type="none"):
    _sum = sum(counter_object.values())
    dist = {k: float(f"{100 * v / _sum:.2f}") for k, v in counter_object.items()}
    if sort_type == "value":
        dist = OrderedDict(sorted(dist.items(), reverse=True))
    return dist


def get_show_name(vid_name):
    """
    get tvshow name from vid_name
    :param vid_name: video clip name
    :return: tvshow name
    """
    show_list = ["friends", "met", "castle", "house", "grey"]
    vid_name_prefix = vid_name.split("_")[0]
    show_name = vid_name_prefix if vid_name_prefix in show_list else "bbt"
    return show_name


def get_abspaths_by_ext(dir_path, ext=(".jpg",)):
    """Get absolute paths to files in dir_path with extensions specified by ext.
    Note this function does work recursively.
    """
    if isinstance(ext, list):
        ext = tuple(ext)
    if isinstance(ext, str):
        ext = tuple(
            [
                ext,
            ]
        )
    filepaths = [
        os.path.join(root, name)
        for root, dirs, files in os.walk(dir_path)
        for name in files
        if name.endswith(tuple(ext))
    ]
    return filepaths


def get_basename_no_ext(path):
    """'/data/movienet/240p_keyframe_feats/tt7672188.npz' --> 'tt7672188'"""
    return os.path.splitext(os.path.split(path)[1])[0]


def dict_to_markdown(d, max_str_len=120):
    # convert list into its str representation
    d = {k: v.__repr__() if isinstance(v, list) else v for k, v in d.items()}
    # truncate string that is longer than max_str_len
    if max_str_len is not None:
        d = {k: v[-max_str_len:] if isinstance(v, str) else v for k, v in d.items()}
    return pd.DataFrame(d, index=[0]).transpose().to_markdown()

In [ ]:
preprocessd_sample = load_jsonl(
    str(PREPROCESSED_DIR / "clotho_moment_train_release.jsonl")
)

preprocessd_sample[0]

In [ ]:
# y_sample, sr_sample = librosa.load(
#     str(TRAIN_DIR / "Venice_40_640.wav"),
#     sr=None
# )

y_sample, sr_sample = torchaudio.load(str(TRAIN_DIR / "Venice_40_640.wav"))

In [ ]:
display(Audio(data=y_sample, rate=sr_sample))

In [ ]:
plt.figure()
librosa.display.waveshow(y_sample, sr=sr_sample, alpha=0.7)
plt.title(f"Waveform – qid {preprocessd_sample[0]['qid']}", fontsize=14, weight="bold")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.show()

In [ ]:
# n_fft = 1024
# hop_length = 256

# mel_spec = librosa.feature.melspectrogram(
#     y=y_sample,
#     sr=sr_sample,
#     n_fft=n_fft,
#     hop_length=hop_length,
#     n_mels=40,
#     fmax=sr_sample / 2,
# )
# log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
# plt.figure()
# librosa.display.specshow(
#     log_mel_spec,
#     sr=sr_sample,
#     hop_length=hop_length,
#     x_axis="time",
#     y_axis="mel",
#     cmap="viridis",
# )
# plt.title("Log‑Mel Spectrogram", fontsize=14, weight="bold")
# plt.colorbar(format="%+2.0f dB")
# plt.tight_layout()
# plt.show()

### Span Utils

In [11]:
def span_xx_to_cxw(xx_spans):
    """
    Args:
        xx_spans: tensor, (#windows, 2) or (..., 2), each row is a window of format (st, ed)

    Returns:
        cxw_spans: tensor, (#windows, 2), each row is a window of format (center=(st+ed)/2, width=(ed-st))
    >>> spans = torch.Tensor([[0, 1], [0.2, 0.4]])
    >>> span_xx_to_cxw(spans)
    tensor([[0.5000, 1.0000],
        [0.3000, 0.2000]])
    >>> spans = torch.Tensor([[[0, 1], [0.2, 0.4]]])
    >>> span_xx_to_cxw(spans)
    tensor([[[0.5000, 1.0000],
         [0.3000, 0.2000]]])
    """
    center = xx_spans.sum(-1) * 0.5
    width = xx_spans[..., 1] - xx_spans[..., 0]
    return torch.stack([center, width], dim=-1)


def span_cxw_to_xx(cxw_spans):
    """
    Args:
        cxw_spans: tensor, (#windows, 2) or (..., 2), the last dim is a row denoting a window of format (center, width)

    >>> spans = torch.Tensor([[0.5000, 1.0000], [0.3000, 0.2000]])
    >>> span_cxw_to_xx(spans)
    tensor([[0.0000, 1.0000],
        [0.2000, 0.4000]])
    >>> spans = torch.Tensor([[[0.5000, 1.0000], [0.3000, 0.2000]]])
    >>> span_cxw_to_xx(spans)
    tensor([[[0.0000, 1.0000],
        [0.2000, 0.4000]]])
    """
    x1 = cxw_spans[..., 0] - 0.5 * cxw_spans[..., 1]
    x2 = cxw_spans[..., 0] + 0.5 * cxw_spans[..., 1]
    return torch.stack([x1, x2], dim=-1)


def temporal_iou(spans1, spans2):
    """
    Args:
        spans1: (N, 2) torch.Tensor, each row defines a span [st, ed]
        spans2: (M, 2) torch.Tensor, ...

    Returns:
        iou: (N, M) torch.Tensor
        union: (N, M) torch.Tensor
    >>> test_spans1 = torch.Tensor([[0, 0.2], [0.5, 1.0]])
    >>> test_spans2 = torch.Tensor([[0, 0.3], [0., 1.0]])
    >>> temporal_iou(test_spans1, test_spans2)
    (tensor([[0.6667, 0.2000],
         [0.0000, 0.5000]]),
     tensor([[0.3000, 1.0000],
             [0.8000, 1.0000]]))
    """
    areas1 = spans1[:, 1] - spans1[:, 0]  # (N, )
    areas2 = spans2[:, 1] - spans2[:, 0]  # (M, )

    left = torch.max(spans1[:, None, 0], spans2[:, 0])  # (N, M)
    right = torch.min(spans1[:, None, 1], spans2[:, 1])  # (N, M)

    inter = (right - left).clamp(min=0)  # (N, M)
    union = areas1[:, None] + areas2 - inter  # (N, M)

    iou = inter / union
    return iou, union


def temporal_intersection_over_pred(gt_spans, pred_spans):
    """intersection over the second input spans
    Args:
        gt_spans: (N, 2),
        pred_spans: (M, 2)

    Returns:

    """
    left = torch.max(gt_spans[:, None, 0], pred_spans[:, 0])
    right = torch.min(gt_spans[:, None, 1], pred_spans[:, 1])

    inter = (right - left).clamp(min=0)  # (N, M)
    inter_over_pred = inter / (pred_spans[:, 1] - pred_spans[:, 0])
    return inter_over_pred


def generalized_temporal_iou(spans1, spans2):
    """
    Generalized IoU from https://giou.stanford.edu/
    Also reference to DETR implementation of generalized_box_iou
    https://github.com/facebookresearch/detr/blob/master/util/box_ops.py#L40

    Args:
        spans1: (N, 2) torch.Tensor, each row defines a span in xx format [st, ed]
        spans2: (M, 2) torch.Tensor, ...

    Returns:
        giou: (N, M) torch.Tensor

    >>> test_spans1 = torch.Tensor([[0, 0.2], [0.5, 1.0]])
    >>> test_spans2 = torch.Tensor([[0, 0.3], [0., 1.0]])
    >>> generalized_temporal_iou(test_spans1, test_spans2)
    tensor([[ 0.6667,  0.2000],
        [-0.2000,  0.5000]])
    """
    spans1 = spans1.float()
    spans2 = spans2.float()
    assert (spans1[:, 1] >= spans1[:, 0]).all()
    assert (spans2[:, 1] >= spans2[:, 0]).all()
    iou, union = temporal_iou(spans1, spans2)

    left = torch.min(spans1[:, None, 0], spans2[:, 0])  # (N, M)
    right = torch.max(spans1[:, None, 1], spans2[:, 1])  # (N, M)
    enclosing_area = (right - left).clamp(min=0)  # (N, M)

    return iou - (enclosing_area - union) / enclosing_area


def generalized_temporal_iou_(spans1, spans2):
    """
    Generalized IoU from https://giou.stanford.edu/
    Also reference to DETR implementation of generalized_box_iou
    https://github.com/facebookresearch/detr/blob/master/util/box_ops.py#L40

    Args:
        spans1: (N, 2) torch.Tensor, each row defines a span in xx format [st, ed]
        spans2: (M, 2) torch.Tensor, ...

    Returns:
        giou: (N, M) torch.Tensor

    >>> test_spans1 = torch.Tensor([[0, 0.2], [0.5, 1.0]])
    >>> test_spans2 = torch.Tensor([[0, 0.3], [0., 1.0]])
    >>> generalized_temporal_iou(test_spans1, test_spans2)
    tensor([[ 0.6667,  0.2000],
        [-0.2000,  0.5000]])
    """
    spans1 = spans1.float()
    spans2 = spans2.float()
    iou, union = temporal_iou(spans1, spans2)

    left = torch.min(spans1[:, None, 0], spans2[:, 0])  # (N, M)
    right = torch.max(spans1[:, None, 1], spans2[:, 1])  # (N, M)
    enclosing_area = (right - left).clamp(min=0)  # (N, M)

    return iou - (enclosing_area - union) / enclosing_area

### Tensor Utils

In [12]:
def pad_sequences_1d(
    sequences, dtype=torch.long, device=torch.device("cpu"), fixed_length=None
):
    """Pad a single-nested list or a sequence of n-d array (torch.tensor or np.ndarray)
    into a (n+1)-d array, only allow the first dim has variable lengths.
    Args:
        sequences: list(n-d tensor or list)
        dtype: np.dtype or torch.dtype
        device:
        fixed_length: pad all seq in sequences to fixed length. All seq should have a length <= fixed_length.
            return will be of shape [len(sequences), fixed_length, ...]
    Returns:
        padded_seqs: ((n+1)-d tensor) padded with zeros
        mask: (2d tensor) of the same shape as the first two dims of padded_seqs,
              1 indicate valid, 0 otherwise
    Examples:
        >>> test_data_list = [[1,2,3], [1,2], [3,4,7,9]]
        >>> pad_sequences_1d(test_data_list, dtype=torch.long)
        >>> test_data_3d = [torch.randn(2,3,4), torch.randn(4,3,4), torch.randn(1,3,4)]
        >>> pad_sequences_1d(test_data_3d, dtype=torch.float)
        >>> test_data_list = [[1,2,3], [1,2], [3,4,7,9]]
        >>> pad_sequences_1d(test_data_list, dtype=np.float32)
        >>> test_data_3d = [np.random.randn(2,3,4), np.random.randn(4,3,4), np.random.randn(1,3,4)]
        >>> pad_sequences_1d(test_data_3d, dtype=np.float32)
    """
    if isinstance(sequences[0], list):
        if "torch" in str(dtype):
            sequences = [torch.tensor(s, dtype=dtype, device=device) for s in sequences]
        else:
            sequences = [np.asarray(s, dtype=dtype) for s in sequences]

    extra_dims = sequences[0].shape[
        1:
    ]  # the extra dims should be the same for all elements
    lengths = [len(seq) for seq in sequences]
    if fixed_length is not None:
        max_length = fixed_length
    else:
        max_length = max(lengths)
    if isinstance(sequences[0], torch.Tensor):
        assert "torch" in str(dtype), "dtype and input type does not match"
        padded_seqs = torch.zeros(
            (len(sequences), max_length) + extra_dims, dtype=dtype, device=device
        )
        mask = torch.zeros(
            (len(sequences), max_length), dtype=torch.float32, device=device
        )
    else:  # np
        assert "numpy" in str(dtype), "dtype and input type does not match"
        padded_seqs = np.zeros((len(sequences), max_length) + extra_dims, dtype=dtype)
        mask = np.zeros((len(sequences), max_length), dtype=np.float32)

    for idx, seq in enumerate(sequences):
        end = lengths[idx]
        padded_seqs[idx, :end] = seq
        mask[idx, :end] = 1
    return padded_seqs, mask  # , lengths


def pad_sequences_2d(sequences, dtype=torch.long):
    """Pad a double-nested list or a sequence of n-d torch tensor into a (n+1)-d tensor,
        only allow the first two dims has variable lengths
    Args:
        sequences: list(n-d tensor or list)
        dtype: torch.long for word indices / torch.float (float32) for other cases
    Returns:
    Examples:
        >>> test_data_list = [[[1, 3, 5], [3, 7, 4, 1]], [[98, 34, 11, 89, 90], [22], [34, 56]],]
        >>> pad_sequences_2d(test_data_list, dtype=torch.long)  # torch.Size([2, 3, 5])
        >>> test_data_3d = [torch.randn(2,2,4), torch.randn(4,3,4), torch.randn(1,5,4)]
        >>> pad_sequences_2d(test_data_3d, dtype=torch.float)  # torch.Size([2, 3, 5])
        >>> test_data_3d2 = [[torch.randn(2,4), ], [torch.randn(3,4), torch.randn(5,4)]]
        >>> pad_sequences_2d(test_data_3d2, dtype=torch.float)  # torch.Size([2, 3, 5])
    # TODO add support for numpy array
    """
    bsz = len(sequences)
    para_lengths = [len(seq) for seq in sequences]
    max_para_len = max(para_lengths)
    sen_lengths = [[len(word_seq) for word_seq in seq] for seq in sequences]
    max_sen_len = max([max(e) for e in sen_lengths])

    if isinstance(sequences[0], torch.Tensor):
        extra_dims = sequences[0].shape[2:]
    elif isinstance(sequences[0][0], torch.Tensor):
        extra_dims = sequences[0][0].shape[1:]
    else:
        sequences = [
            [torch.Tensor(word_seq, dtype=dtype) for word_seq in seq]
            for seq in sequences
        ]
        extra_dims = ()

    padded_seqs = torch.zeros(
        (bsz, max_para_len, max_sen_len) + extra_dims, dtype=dtype
    )
    mask = torch.zeros(bsz, max_para_len, max_sen_len).float()

    for b_i in range(bsz):
        for sen_i, sen_l in enumerate(sen_lengths[b_i]):
            padded_seqs[b_i, sen_i, :sen_l] = sequences[b_i][sen_i]
            mask[b_i, sen_i, :sen_l] = 1
    return padded_seqs, mask  # , sen_lengths

In [13]:
# Example

test_data_list = [[1, 2, 3], [1, 2], [3, 4, 7, 9]]
pad_sequences_1d(test_data_list, dtype=torch.long)

(tensor([[1, 2, 3, 0],
         [1, 2, 0, 0],
         [3, 4, 7, 9]]),
 tensor([[1., 1., 1., 0.],
         [1., 1., 0., 0.],
         [1., 1., 1., 1.]]))

In [ ]:
# Ref: https://github.com/nttcslab/m2d/tree/master/examples

# !pip install -q einops nnAudio
# !wget https://raw.githubusercontent.com/nttcslab/m2d/master/examples/portable_m2d.py
# !wget https://github.com/nttcslab/m2d/releases/download/v0.5.0/m2d_clap_vit_base-80x1001p16x16p16kpBpTI-2025.zip

# with zipfile.ZipFile("m2d_clap_vit_base-80x1001p16x16p16kpBpTI-2025.zip", "r") as zip_ref:
#     zip_ref.extractall(".")

# !find m2d_clap_vit_base-80x1001p16x16p16kpBpTI-2025 -name *.pth

In [ ]:
# from portable_m2d import PortableM2D
# model = PortableM2D(weight_file='m2d_clap_vit_base-80x1001p16x16p16kpBpTI-2025/checkpoint-30.pth')
# model.eval()

# # model.to(DEVICE)

# audio = torch.from_numpy(y_sample).unsqueeze(0)
# embedding = model(audio)

# embedding # [1, 513, 3840] => [:, : 768 * 5]

In [ ]:
# # clap_model_id = "laion/clap-htsat-unfused"
# clap_model_id = "laion/clap-htsat-fused"
# clap_model = AutoModel.from_pretrained(clap_model_id).to(DEVICE)
# # clap_tokenizer = AutoTokenizer.from_pretrained(clap_model_id)
# clap_processor = AutoProcessor.from_pretrained(clap_model_id)

# clap_processor.feature_extractor = ClapFeatureExtractor(
#     sampling_rate=sr_sample
# )

### Data Preparing

Include 3 steps:

1. DataProcessor: Imply models for features extraction, augmentations, ...
2. Dataset: Use DataProcessor, defining input and output of the model
3. DataLoader: Use Dataset, load data to GPU, ..


#### Dataset

1. Prepare Dataset
2. Prepare Dataloader

In [14]:
class BaseOptions(object):
    def __init__(self, config_path):
        self.config_path = config_path
        self.opt = {}

    @property
    def option(self):
        if len(self.opt) == 0:
            raise RuntimeError("option is empty. Did you run parse()?")
        return self.opt

    def update(self, yaml_file):
        with open(yaml_file, "r") as f:
            yml = yaml.load(f, Loader=yaml.FullLoader)
            self.opt.update(yml)

    def parse(self):
        with open(self.config_path, "r") as f:
            yml = yaml.load(f, Loader=yaml.FullLoader)
            self.opt.update(yml)

        self.opt = EasyDict(self.opt)
        self.opt.ckpt_filepath = os.path.join(
            self.opt.results_dir, self.opt.ckpt_filename
        )
        self.opt.train_log_filepath = os.path.join(
            self.opt.results_dir, self.opt.train_log_filename
        )
        self.opt.eval_log_filepath = os.path.join(
            self.opt.results_dir, self.opt.eval_log_filename
        )

In [15]:
def get_dir_size(path="."):
    total_size = 0
    try:
        with os.scandir(path) as it:
            for entry in it:
                if entry.is_file():
                    # entry.stat() is cached on some systems, making this very fast
                    total_size += entry.stat().st_size
                elif entry.is_dir():
                    # Recursively call the function for subdirectories
                    total_size += get_dir_size(entry.path)
    except PermissionError:
        # Handle folders you don't have access to
        return 0
    return total_size


def format_size(bytes):
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if bytes < 1024:
            return f"{bytes:.2f} {unit}"
        bytes /= 1024


def _copy_single_file(src_file, root_src, dst_dir):
    """Helper function to copy a single file. (Runs inside the thread)"""
    # Calculate the relative path to maintain folder structure
    relative_path = src_file.relative_to(root_src)
    dest_item = dst_dir / relative_path

    # Ensure the destination subdirectory exists
    dest_item.parent.mkdir(parents=True, exist_ok=True)

    # Copy the file along with its metadata
    shutil.copy2(src_file, dest_item)
    return True


def copy_dir_with_progress(src, dst, max_workers=16):
    """
    Copies a directory recursively using multiple threads with a tqdm progress bar.
    """
    src_path = Path(src)
    dst_path = Path(dst)

    if not src_path.exists():
        print(f"Error: Source directory '{src}' does not exist.")
        return

    # 1. Scan and count all files
    print(f"Scanning '{src}' for files...")
    all_files = [f for f in src_path.rglob("*") if f.is_file()]
    total_files = len(all_files)

    if total_files == 0:
        print("No files found to copy.")
        return

    print(
        f"Found {total_files} files. Starting multithreaded copy with {max_workers} workers..."
    )

    # 2. Set up the Thread Pool
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all the copy tasks to the thread pool
        futures = {
            executor.submit(_copy_single_file, item, src_path, dst_path): item
            for item in all_files
        }

        # 3. Update the tqdm progress bar as each thread completes its task
        for future in tqdm(
            concurrent.futures.as_completed(futures),
            total=total_files,
            desc="Copying Data",
            unit="file",
        ):
            try:
                future.result()  # This will raise an exception if the thread failed
            except Exception as e:
                print(f"Error copying {futures[future].name}: {e}")

In [ ]:
# # NOTE: Move dir from Drive to Colab for faster execution

# copy_dir_with_progress(
#     src=str(FEATURES_DIR),
#     dst="/content"
# )


%timeit

print("Copy feature dir")


!cp -r /content/drive/MyDrive/Dataset/DCASE2026/CLOTHO-MOMENT/features.zip /content/

!unzip /content/features.zip

print("Copy pre-processed dir")

!cp -r /content/drive/MyDrive/Dataset/DCASE2026/CLOTHO-MOMENT/preprocessed /content/

In [16]:
class StartEndDataset(Dataset):
    """One line in data loaded from data_path."
    {
      "qid": 7803,
      "query": "Man in gray top walks from outside to inside.",
      "duration": 150,
      "vid": "RoripwjYFp8_360.0_510.0",
      "relevant_clip_ids": [13, 14, 15, 16, 17],
      "relevant_windows": [[26, 36]]
    }
    """

    def __init__(
        self,
        data_path: str,
        a_feat_dir: str,
        q_feat_dir: str,
        q_feat_type: str = "last_hidden_state",
        a_feat_type: str = "pann",
        max_q_l: int = 32,
        max_a_l: int = 75,
        ctx_mode: str = "video",
        clip_len: int = 2,
        max_windows: int = 5,
        span_loss_type: str = "l1",
        load_labels: bool = True,
    ) -> None:
        self.data_path = data_path
        self.a_feat_dir = a_feat_dir
        self.q_feat_dir = q_feat_dir
        self.q_feat_type = q_feat_type
        self.a_feat_type = a_feat_type

        if max_a_l == -1:
            max_a_l = 100000000

        if max_q_l == -1:
            max_q_l = 100

        self.max_q_l = max_q_l
        self.max_a_l = max_a_l

        self.ctx_mode = ctx_mode
        self.use_tef = "tef" in ctx_mode
        self.use_audio = "audio" in ctx_mode
        self.clip_len = clip_len
        self.max_windows = max_windows  # maximum number of windows to use as labels
        self.span_loss_type = span_loss_type
        self.load_labels = load_labels
        self.data = self.load_data()

    def load_data(self) -> List[Dict[str, Any]]:
        datalist = load_jsonl(self.data_path)
        return datalist

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, index: int) -> Dict[str, Any]:
        meta = self.data[index]

        model_inputs = dict()

        model_inputs["query_feat"] = self._get_query_feat_by_qid(
            meta["qid"]
        )  # (Dq, ) or (Lq, Dq)
        model_inputs["audio_feat"] = self._get_audio_feat_by_vid(meta["vid"])
        ctx_l = len(model_inputs["audio_feat"])

        if self.use_tef:
            duration = meta["duration"]  # Total video duration in seconds
            clip_indices = torch.arange(0, ctx_l, 1.0)  # [0, 1, 2, ..., ctx_l-1]
            tef_st = (clip_indices * self.clip_len) / duration  # Normalized start times
            tef_ed = (
                (clip_indices + 1) * self.clip_len
            ) / duration  # Normalized end times
            tef_ed = torch.clamp(tef_ed, max=1.0)  # Ensure it doesn't exceed 1.0
            tef = torch.stack([tef_st, tef_ed], dim=1)  # (ctx_l, 2)
            model_inputs["audio_feat"] = torch.cat(
                [model_inputs["audio_feat"], tef], dim=1
            )
        if self.load_labels:
            model_inputs["span_labels"] = self.get_span_labels(
                meta["relevant_windows"], ctx_l, meta["duration"]
            )
            (
                model_inputs["saliency_pos_labels"],
                model_inputs["saliency_neg_labels"],
                model_inputs["saliency_all_labels"],
            ) = self.get_saliency_labels_sub_as_query(
                meta["relevant_windows"][0], ctx_l
            )

        return dict(meta=meta, model_inputs=model_inputs)

    def get_span_labels(
        self, windows: List[List[float]], ctx_l: int, duration: float
    ) -> torch.Tensor:
        """
        windows: list([st, ed]) in seconds. E.g. [[26, 36]], corresponding st_ed clip_indices [[13, 17]] (inclusive)
            Note a maximum of `self.max_windows` windows are used.
        returns Tensor of shape (#windows, 2), each row is [center, width] normalized by video length
        """
        if len(windows) > self.max_windows:
            random.shuffle(windows)
            windows = windows[: self.max_windows]
        if self.span_loss_type == "l1":
            windows = torch.Tensor(windows) / duration  # normalized windows in xx
            windows = span_xx_to_cxw(windows)  # normalized windows in cxw
        elif self.span_loss_type == "ce":
            windows = torch.Tensor(
                [
                    [
                        int(w[0] / self.clip_len),
                        min(int(w[1] / self.clip_len), ctx_l) - 1,
                    ]
                    for w in windows
                ]
            ).long()  # inclusive
        else:
            raise NotImplementedError
        return windows

    def get_saliency_labels_sub_as_query(
        self, gt_window: List[float], ctx_l: int, max_n: int = 2
    ) -> Tuple[List[int], List[int], np.ndarray]:
        gt_st = int(gt_window[0] / self.clip_len)
        gt_ed = max(0, min(int(gt_window[1] / self.clip_len), ctx_l) - 1)

        if gt_st > gt_ed:
            gt_st = gt_ed

        if gt_st != gt_ed:
            pos_clip_indices = random.sample(range(gt_st, gt_ed + 1), k=max_n)
        else:
            pos_clip_indices = [gt_st, gt_st]

        neg_pool = list(range(0, gt_st)) + list(
            range(gt_ed + 1, ctx_l)
        )  # to fix bugs / works..?
        try:
            neg_clip_indices = random.sample(neg_pool, k=max_n)
        except:
            neg_clip_indices = pos_clip_indices

        score_array = np.zeros(ctx_l)
        score_array[gt_st : gt_ed + 1] = 1

        return pos_clip_indices, neg_clip_indices, score_array

    def get_saliency_labels(
        self,
        rel_clip_ids: List[int],
        scores: List[List[float]],
        ctx_l: int,
        max_n: int = 1,
        add_easy_negative: bool = True,
    ) -> Tuple[List[int], List[int]]:
        """Sum the scores from the three annotations, then take the two clips with the
        maximum scores as positive, and two with the minimum scores as negative.
        Args:
            rel_clip_ids: list(int), list of relevant clip ids
            scores: list([anno1_score, anno2_score, anno3_score]),
            ctx_l: int
            max_n: int, #clips to use as positive and negative, for easy and hard negative, respectively.
            add_easy_negative: bool, if True, sample eay negative outside the relevant_clip_ids.
        """
        # indices inside rel_clip_ids
        scores = np.array(scores)  # (#rel_clips, 3)
        agg_scores = np.sum(scores, 1)  # (#rel_clips, )
        sort_indices = np.argsort(agg_scores)  # increasing

        # indices in the whole video
        # the min(_, ctx_l-1) here is incorrect, but should not cause
        # much troubles since this should be rarely used.
        hard_pos_clip_indices = [
            min(rel_clip_ids[idx], ctx_l - 1) for idx in sort_indices[-max_n:]
        ]
        hard_neg_clip_indices = [
            min(rel_clip_ids[idx], ctx_l - 1) for idx in sort_indices[:max_n]
        ]
        easy_pos_clip_indices = []
        easy_neg_clip_indices = []
        if add_easy_negative:
            easy_neg_pool = list(set(range(ctx_l)) - set(rel_clip_ids))
            if len(easy_neg_pool) >= max_n:
                easy_pos_clip_indices = random.sample(rel_clip_ids, k=max_n)
                easy_neg_clip_indices = random.sample(easy_neg_pool, k=max_n)
            else:  # copy the hard ones
                easy_pos_clip_indices = hard_pos_clip_indices
                easy_neg_clip_indices = hard_neg_clip_indices

        pos_clip_indices = hard_pos_clip_indices + easy_pos_clip_indices
        neg_clip_indices = hard_neg_clip_indices + easy_neg_clip_indices
        return pos_clip_indices, neg_clip_indices

    def _get_query_feat_by_qid(self, qid: int) -> np.ndarray:
        q_feat_path = join(self.q_feat_dir, f"qid{qid}.npz")
        q_feat = np.load(q_feat_path)["last_hidden_state"]
        return q_feat

    def _get_audio_feat_by_vid(self, vid: str) -> torch.Tensor:
        _feat_path = join(self.a_feat_dir, f"{vid}.npz")
        _feat = np.load(_feat_path)["features"][: self.max_a_l].astype(np.float32)
        _feat = l2_normalize_np_array(_feat)
        return torch.from_numpy(_feat)

In [17]:
def start_end_collate(
    batch: List[Dict[str, Any]],
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    batch_meta = [e["meta"] for e in batch]

    model_inputs_keys = batch[0]["model_inputs"].keys()
    batched_data = dict()
    for k in model_inputs_keys:
        if k == "span_labels":
            batched_data[k] = [
                dict(spans=e["model_inputs"]["span_labels"]) for e in batch
            ]
            continue
        if k in ["saliency_pos_labels", "saliency_neg_labels"]:
            batched_data[k] = torch.LongTensor([e["model_inputs"][k] for e in batch])
            continue
        if k == "saliency_all_labels":
            pad_data, mask_data = pad_sequences_1d(
                [e["model_inputs"][k] for e in batch],
                dtype=np.float32,
                fixed_length=None,
            )
            batched_data[k] = torch.tensor(pad_data, dtype=torch.float32)
            continue

        if batch[0]["model_inputs"][k].dtype == torch.float32:
            batched_data[k] = pad_sequences_1d(
                [e["model_inputs"][k] for e in batch],
                dtype=torch.float32,
                fixed_length=None,
            )
        else:
            batched_data[k] = pad_sequences_1d(
                [torch.from_numpy(e["model_inputs"][k]) for e in batch],
                dtype=torch.float32,
                fixed_length=None,
            )
    return batch_meta, batched_data


def prepare_batch_inputs(
    batched_model_inputs: Dict[str, Any],
    device: torch.device,
    non_blocking: bool = False,
) -> Tuple[Dict[str, torch.Tensor], Optional[Dict[str, Any]]]:
    model_inputs = dict(
        src_txt=batched_model_inputs["query_feat"][0].to(
            device, non_blocking=non_blocking
        ),
        src_txt_mask=batched_model_inputs["query_feat"][1].to(
            device, non_blocking=non_blocking
        ),
    )

    if "audio_feat" in batched_model_inputs:
        model_inputs["src_aud"] = batched_model_inputs["audio_feat"][0].to(
            device, non_blocking=non_blocking
        )
        model_inputs["src_aud_mask"] = batched_model_inputs["audio_feat"][1].to(
            device, non_blocking=non_blocking
        )

    targets = {}
    if "span_labels" in batched_model_inputs:
        targets["span_labels"] = [
            dict(spans=e["spans"].to(device, non_blocking=non_blocking))
            for e in batched_model_inputs["span_labels"]
        ]
    if "saliency_pos_labels" in batched_model_inputs:
        for name in ["saliency_pos_labels", "saliency_neg_labels"]:
            targets[name] = batched_model_inputs[name].to(
                device, non_blocking=non_blocking
            )

    if "saliency_all_labels" in batched_model_inputs:
        targets["saliency_all_labels"] = batched_model_inputs["saliency_all_labels"].to(
            device, non_blocking=non_blocking
        )

    targets = None if len(targets) == 0 else targets
    return model_inputs, targets

In [94]:
# config_path = PREPROCESSED_DIR / "train_config_clotho.yml"
# opt = read_yaml(config_path)
# opt


config_path = LOCAL_DIR / "config" / "config_pretraining.yml"
opt = read_yaml(config_path)
opt

{'seed': 2023,
 'device': 'mps',
 'num_workers': 0,
 'lr': 0.0001,
 'lr_drop': 400,
 'wd': 0.0001,
 'n_epoch': 200,
 'bsz': 32,
 'eval_bsz': 100,
 'grad_clip': 0.1,
 'max_windows': 5,
 'clip_length': 1,
 'eval_epoch_interval': 1,
 'position_embedding': 'sine',
 'enc_layers': 2,
 'dec_layers': 2,
 'dim_feedforward': 1024,
 'hidden_dim': 256,
 'input_dropout': 0.5,
 'dropout': 0.1,
 'nheads': 8,
 'num_queries': 10,
 'n_input_proj': 2,
 'saliency_margin': 0.2,
 'span_loss_type': 'l1',
 'set_cost_span': 10,
 'set_cost_giou': 1,
 'set_cost_class': 4,
 'span_loss_coef': 10,
 'giou_loss_coef': 1,
 'label_loss_coef': 4,
 'eos_coef': 0.1,
 'lw_saliency': 1,
 'ckpt_filename': 'best_checkpoint.pth',
 'train_log_filename': 'train.log',
 'eval_log_filename': 'val.log',
 'eval_split_name': 'val',
 'aux_loss': True,
 'model_ema': False,
 'ema_decay': 0.9,
 'results_dir': 'results_pretraining',
 'dset_name': 'clotho-moment',
 'train_path': 'data/clotho_moment_train_release.jsonl',
 'val_path': 'data/c

In [ ]:
!cp -r /content/drive/MyDrive/Dataset/DCASE2026/CLOTHO-MOMENT/preprocessed /content/

In [95]:
opt["train_path"] = str(PREPROCESSED_DIR / "local_training_data.jsonl")
opt["val_path"] = str(PREPROCESSED_DIR / "clotho_moment_valid_val.jsonl")
opt["test_path"] = str(PREPROCESSED_DIR / "clotho_moment_valid_test.jsonl")


# opt['a_feat_dir'] = str(FEATURES_DIR / "clotho-moment" / "clap")
# opt['t_feat_dir'] = str(FEATURES_DIR / "clotho-moment" / "clap_text")


a_feat_dir = str(DATA_DIR / "clotho-moment" / "clap")
q_feat_dir = str(DATA_DIR / "clotho-moment" / "clap_text")
opt["a_feat_dir"] = a_feat_dir
opt["t_feat_dir"] = q_feat_dir

In [ ]:
# a_feat_dir = str(Path("/content") / "clotho-moment" / "clap")
# q_feat_dir = str(Path("/content") / "clotho-moment" / "clap_text")


# q_np_feat = np.load(
#     os.path.join(q_feat_dir, os.listdir(q_feat_dir)[0])
# )


# q_np_feat['last_hidden_state'].shape


# a_np_feat = np.load(
#     os.path.join(a_feat_dir, os.listdir(a_feat_dir)[3])
# )


# a_np_feat.keys()

# a_np_feat['features'].shape

# a_np_feat['proj_features'].shape

In [135]:
dataset_config = EasyDict(
    data_path=opt.get("train_path"),
    ctx_mode=opt.get("ctx_mode"),
    a_feat_dir=opt.get("a_feat_dir"),
    q_feat_dir=opt.get("t_feat_dir"),
    q_feat_type="last_hidden_state",
    a_feat_type=opt.get("a_feat_type"),
    max_q_l=opt.get("max_q_l"),
    max_a_l=opt.get("max_a_l"),
    clip_len=opt.get("clip_length"),
    max_windows=opt.get("max_windows"),
    span_loss_type=opt.get("span_loss_type"),
    load_labels=True,
)

In [133]:
train_dataset = StartEndDataset(
    **dataset_config,
)

In [134]:
dataset_config

{'data_path': '/Users/minhld/workspace/projects/DCASE_2026/data/preprocessed/local_training_data.jsonl',
 'ctx_mode': 'audio_tef',
 'a_feat_dir': '/Users/minhld/workspace/projects/DCASE_2026/data/clotho-moment/clap',
 'q_feat_dir': '/Users/minhld/workspace/projects/DCASE_2026/data/clotho-moment/clap_text',
 'q_feat_type': 'last_hidden_state',
 'a_feat_type': 'clap',
 'max_q_l': 32,
 'max_a_l': 300,
 'clip_len': 1,
 'max_windows': 5,
 'span_loss_type': 'l1',
 'load_labels': True}

In [ ]:
# import os
# from pathlib import Path

# # 1. Ensure these are Path objects
# PREPROCESSED_DIR = Path(PREPROCESSED_DIR)
# # a_feat_dir = Path(q_feat_dir)

# # 2. Get all existing files in the directory once (Very fast)
# existing_files = set(os.listdir(q_feat_dir))

# global_jsonl = load_jsonl(str(PREPROCESSED_DIR / "clotho_moment_valid_train.jsonl"))

# local_jsonl = []
# for data in global_jsonl:
#     qid = data['qid']
#     # 3. Check against the set instead of the disk
#     filename = f"qid{qid}.npz"

#     if filename in existing_files:
#         local_jsonl.append(data)

# print(f"Filtered {len(global_jsonl)} down to {len(local_jsonl)} valid samples.")


# output_path = LOCAL_DIR / "config" / "local_training_config.jsonl"
# with open(output_path, 'w', encoding='utf-8') as f:
#     for entry in local_jsonl:
#         # Convert dictionary to JSON string and write it
#         json_record = json.dumps(entry, ensure_ascii=False)
#         f.write(json_record + '\n')
# print(f"Successfully wrote {len(local_jsonl)} lines to {output_path}")

In [136]:
train_sample = train_dataset[10]

train_sample

{'meta': {'qid': '00014',
  'query': 'Water drips lightly as a dishwasher runs a cycle.',
  'duration': 60,
  'vid': 'Venice_17.0_77.0',
  'relevant_windows': [[29.4, 46.4]],
  'fg_dB': -2.8594276217940298},
 'model_inputs': {'query_feat': array([[-0.2671024 ,  0.31409022, -0.20707619, ..., -0.05527776,
          -0.04029597, -0.09613095],
         [ 0.07893793,  0.96753603, -0.5789807 , ..., -0.5021244 ,
          -0.04657622, -0.37173513],
         [-0.05350053,  1.1170663 , -1.2730165 , ..., -0.11704937,
           0.53195167,  0.00814806],
         ...,
         [-0.36259237,  0.44742185,  0.01398961, ...,  0.50016105,
           0.38664854, -0.2378075 ],
         [-0.29393604,  0.5116786 ,  0.02193145, ...,  0.65585214,
           0.47843364, -0.15022379],
         [-0.09246193,  0.46447322, -0.20965803, ...,  0.75279397,
           0.47497916, -0.00817642]], shape=(14, 768), dtype=float32),
  'audio_feat': tensor([[ 5.3610e-02,  6.1957e-02, -6.3998e-05,  ..., -3.5745e-02,
       

### Core Model


In [106]:
def inverse_sigmoid(x: torch.Tensor, eps: float = 1e-3) -> torch.Tensor:
    x = x.clamp(min=0, max=1)
    x1 = x.clamp(min=eps)
    x2 = (1 - x).clamp(min=eps)
    return torch.log(x1 / x2)


def count_parameters(model, verbose=True):
    """Count number of parameters in PyTorch model,
    References: https://discuss.pytorch.org/t/how-do-i-check-the-number-of-parameters-of-a-model/4325/7.
    """
    n_all = sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_frozen = n_all - n_trainable
    if verbose:
        print(
            "Parameter Count: all {:,d}; trainable {:,d}; frozen {:,d}".format(
                n_all, n_trainable, n_frozen
            )
        )
    return n_all, n_trainable

In [25]:
class MLP(nn.Module):
    """Very simple multi-layer perceptron (also called FFN)"""

    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super().__init__()
        self.num_layers = num_layers
        h = [hidden_dim] * (num_layers - 1)
        self.layers = nn.ModuleList(
            nn.Linear(n, k) for n, k in zip([input_dim] + h, h + [output_dim])
        )

    def forward(self, x):
        """Forward pass through the MLP.

        Args:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: Output tensor.
        """
        for i, layer in enumerate(self.layers):
            x = F.relu(layer(x)) if i < self.num_layers - 1 else layer(x)
        return x


class LinearLayer(nn.Module):
    """linear layer configurable with layer normalization, dropout, ReLU."""

    def __init__(self, in_hsz, out_hsz, layer_norm=True, dropout=0.1, relu=True):
        super(LinearLayer, self).__init__()
        self.relu = relu
        self.layer_norm = layer_norm
        if layer_norm:
            self.LayerNorm = nn.LayerNorm(in_hsz)
        layers = [nn.Dropout(dropout), nn.Linear(in_hsz, out_hsz)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        """(N, L, D)"""
        if self.layer_norm:
            x = self.LayerNorm(x)
        x = self.net(x)
        if self.relu:
            x = F.relu(x, inplace=True)
        return x  # (N, L, D)

#### Positional Encoding

In [26]:
class TrainablePositionalEncoding(nn.Module):
    """Construct the embeddings from word, position and token_type embeddings."""

    def __init__(self, max_position_embeddings, hidden_size, dropout=0.1):
        super(TrainablePositionalEncoding, self).__init__()
        self.position_embeddings = nn.Embedding(max_position_embeddings, hidden_size)
        self.LayerNorm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_feat):
        """
        Args:
            input_feat: (N, L, D)
        """
        bsz, seq_length = input_feat.shape[:2]
        position_ids = torch.arange(
            seq_length, dtype=torch.long, device=input_feat.device
        )
        position_ids = position_ids.unsqueeze(0).repeat(bsz, 1)  # (N, L)

        position_embeddings = self.position_embeddings(position_ids)

        embeddings = self.LayerNorm(input_feat + position_embeddings)
        embeddings = self.dropout(embeddings)
        return embeddings


class PositionEmbeddingSine(nn.Module):
    """
    This is a more standard version of the position embedding, very similar to the one
    used by the Attention is all you need paper, generalized to work on images. (To 1D sequences)
    """

    def __init__(
        self, num_pos_feats=64, temperature=10000, normalize=False, scale=None
    ):
        super().__init__()
        self.num_pos_feats = num_pos_feats
        self.temperature = temperature
        self.normalize = normalize
        if scale is not None and normalize is False:
            raise ValueError("normalize should be True if scale is passed")
        if scale is None:
            scale = 2 * math.pi
        self.scale = scale

    def forward(self, x, mask):
        """
        Args:
            x: torch.tensor, (batch_size, L, d)
            mask: torch.tensor, (batch_size, L), with 1 as valid

        Returns:

        """
        assert mask is not None
        x_embed = mask.cumsum(1, dtype=torch.float32)  # (bsz, L)
        if self.normalize:
            eps = 1e-6
            x_embed = x_embed / (x_embed[:, -1:] + eps) * self.scale

        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.num_pos_feats)

        pos_x = x_embed[:, :, None] / dim_t  # (bsz, L, num_pos_feats)
        pos_x = torch.stack(
            (pos_x[:, :, 0::2].sin(), pos_x[:, :, 1::2].cos()), dim=3
        ).flatten(2)  # (bsz, L, num_pos_feats*2)
        return pos_x  # .permute(0, 2, 1)  # (bsz, num_pos_feats*2, L)


class PositionEmbeddingLearned(nn.Module):
    """
    Absolute pos embedding, learned.
    """

    def __init__(self, num_pos_feats=256):
        super().__init__()
        self.row_embed = nn.Embedding(50, num_pos_feats)
        self.col_embed = nn.Embedding(50, num_pos_feats)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.uniform_(self.row_embed.weight)
        nn.init.uniform_(self.col_embed.weight)

    def forward(self, x, mask):
        h, w = x.shape[-2:]
        i = torch.arange(w, device=x.device)
        j = torch.arange(h, device=x.device)
        x_emb = self.col_embed(i)
        y_emb = self.row_embed(j)
        pos = (
            torch.cat(
                [
                    x_emb.unsqueeze(0).repeat(h, 1, 1),
                    y_emb.unsqueeze(1).repeat(1, w, 1),
                ],
                dim=-1,
            )
            .permute(2, 0, 1)
            .unsqueeze(0)
            .repeat(x.shape[0], 1, 1, 1)
        )
        return pos


def build_position_encoding(args):
    N_steps = args.hidden_dim
    if args.position_embedding in ("v2", "sine"):
        # TODO find a better way of exposing other arguments
        position_embedding = PositionEmbeddingSine(N_steps, normalize=True)
    # elif args.position_embedding in ('v3', 'learned'):
    #     position_embedding = PositionEmbeddingLearned(N_steps)
    else:
        raise ValueError(f"not supported {args.position_embedding}")

    txt_pos_embed = TrainablePositionalEncoding(
        max_position_embeddings=args.max_q_l,
        hidden_size=args.hidden_dim,
        dropout=args.input_dropout,
    )
    return position_embedding, txt_pos_embed

In [27]:
class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, base: int = 10000):
        super().__init__()
        self.dim = dim
        self.base = base
        # Precompute inverse frequencies
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, x: torch.Tensor, seq_len: int = None):
        """
        x: [batch_size, num_heads, seq_len, head_dim] or [seq_len, ...]
        Returns: cos, sin tensors for rotation
        """
        if seq_len is None:
            seq_len = x.shape[-2]  # Assume last two dims are seq_len, head_dim

        t = torch.arange(seq_len, device=x.device).type_as(self.inv_freq)
        freqs = torch.outer(t, self.inv_freq)  # [seq_len, dim//2]

        # Create cos and sin
        emb = torch.cat((freqs, freqs), dim=-1)  # [seq_len, dim]
        cos = emb.cos()[None, None, :, :]  # [1, 1, seq_len, dim]
        sin = emb.sin()[None, None, :, :]

        return cos, sin


def apply_rotary_emb(
    q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor
):
    """
    Apply RoPE to query and key.
    q, k: [batch, heads, seq_len, head_dim]
    """
    # Reshape for rotation: split head_dim into pairs
    q_ = q.float().reshape(*q.shape[:-1], -1, 2)  # [..., dim//2, 2]
    k_ = k.float().reshape(*k.shape[:-1], -1, 2)

    cos = cos.squeeze(0).squeeze(0)[:, : q_.shape[-2] * 2]  # Adjust if needed
    sin = sin.squeeze(0).squeeze(0)[:, : q_.shape[-2] * 2]
    cos = cos.reshape(1, 1, -1, 2).expand_as(q_)
    sin = sin.reshape(1, 1, -1, 2).expand_as(q_)

    # Rotation: (x1, x2) -> (x1*cos - x2*sin, x1*sin + x2*cos)
    q_out = torch.stack(
        [
            q_[..., 0] * cos[..., 0] - q_[..., 1] * sin[..., 0],
            q_[..., 1] * cos[..., 0] + q_[..., 0] * sin[..., 0],
        ],
        dim=-1,
    ).flatten(-2)

    k_out = torch.stack(
        [
            k_[..., 0] * cos[..., 0] - k_[..., 1] * sin[..., 0],
            k_[..., 1] * cos[..., 0] + k_[..., 0] * sin[..., 0],
        ],
        dim=-1,
    ).flatten(-2)

    return q_out.type_as(q), k_out.type_as(k)


"""
# In your attention forward:
rotary_emb = RotaryEmbedding(head_dim)
cos, sin = rotary_emb(x, seq_len)
q, k = apply_rotary_emb(q, k, cos, sin)
"""

'\n# In your attention forward:\nrotary_emb = RotaryEmbedding(head_dim)\ncos, sin = rotary_emb(x, seq_len)\nq, k = apply_rotary_emb(q, k, cos, sin)\n'

In [28]:
def get_alibi_slopes(n_heads: int):
    """Generate slopes for each head (geometric progression)"""
    start = 2 ** (-8.0 / n_heads)
    return torch.tensor(
        [start * (start**i) for i in range(n_heads)], dtype=torch.float32
    )


def create_alibi_bias(n_heads: int, seq_len: int, device: torch.device):
    """Create the ALiBi bias matrix: [n_heads, seq_len, seq_len]"""
    slopes = get_alibi_slopes(n_heads).to(device).view(n_heads, 1, 1)

    # Create distance matrix (negative for recency bias)
    pos = torch.arange(seq_len, device=device)
    distances = pos.unsqueeze(0) - pos.unsqueeze(1)  # [seq_len, seq_len]

    # Bias = -slope * distance  (only penalize past, but usually full matrix)
    alibi = distances.unsqueeze(0) * slopes  # [n_heads, seq_len, seq_len]
    return alibi  # You add this to attention scores (before softmax)


# Example integration in attention
class AttentionWithALiBi(nn.Module):
    def __init__(self, n_heads: int, head_dim: int):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = head_dim
        self.alibi_slopes = get_alibi_slopes(n_heads)

    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, mask=None):
        # q, k, v: [batch, heads, seq_len, head_dim]
        batch, heads, seq_len, _ = q.shape

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # Add ALiBi bias
        alibi = create_alibi_bias(heads, seq_len, q.device)
        scores = scores + alibi

        if mask is not None:
            scores = scores + mask

        attn = F.softmax(scores, dim=-1)
        output = torch.matmul(attn, v)
        return output

In [29]:
def gen_sineembed_for_position(pos_tensor):
    """Generates sine embeddings for position tensors.

    Args:
        pos_tensor (torch.Tensor): Position tensor.

    Returns:
        torch.Tensor: Sine embeddings.
    """
    # n_query, bs, _ = pos_tensor.size()
    # sineembed_tensor = torch.zeros(n_query, bs, 256)
    scale = 2 * math.pi
    dim_t = torch.arange(128, dtype=torch.float32, device=pos_tensor.device)
    dim_t = 10000 ** (2 * (dim_t // 2) / 128)
    center_embed = pos_tensor[:, :, 0] * scale
    pos_x = center_embed[:, :, None] / dim_t
    pos_x = torch.stack(
        (pos_x[:, :, 0::2].sin(), pos_x[:, :, 1::2].cos()), dim=3
    ).flatten(2)

    span_embed = pos_tensor[:, :, 1] * scale
    pos_w = span_embed[:, :, None] / dim_t
    pos_w = torch.stack(
        (pos_w[:, :, 0::2].sin(), pos_w[:, :, 1::2].cos()), dim=3
    ).flatten(2)

    pos = torch.cat((pos_x, pos_w), dim=2)
    return pos

#### Attention Module


1. Multi-head Attention


Consider

- Mamba-based Cross Attention

In [30]:
def multi_head_attention_forward(
    query: Tensor,
    key: Tensor,
    value: Tensor,
    embed_dim_to_check: int,
    num_heads: int,
    in_proj_weight: Tensor,
    in_proj_bias: Tensor,
    bias_k: Optional[Tensor],
    bias_v: Optional[Tensor],
    add_zero_attn: bool,
    dropout_p: float,
    out_proj_weight: Tensor,
    out_proj_bias: Tensor,
    training: bool = True,
    key_padding_mask: Optional[Tensor] = None,
    need_weights: bool = True,
    attn_mask: Optional[Tensor] = None,
    use_separate_proj_weight: bool = False,
    q_proj_weight: Optional[Tensor] = None,
    k_proj_weight: Optional[Tensor] = None,
    v_proj_weight: Optional[Tensor] = None,
    static_k: Optional[Tensor] = None,
    static_v: Optional[Tensor] = None,
    out_dim: Optional[Tensor] = None,
) -> Tuple[Tensor, Optional[Tensor]]:
    if not torch.jit.is_scripting():
        tens_ops = (
            query,
            key,
            value,
            in_proj_weight,
            in_proj_bias,
            bias_k,
            bias_v,
            out_proj_weight,
            out_proj_bias,
        )
        if any([type(t) is not Tensor for t in tens_ops]) and has_torch_function(
            tens_ops
        ):
            return handle_torch_function(
                multi_head_attention_forward,
                tens_ops,
                query,
                key,
                value,
                embed_dim_to_check,
                num_heads,
                in_proj_weight,
                in_proj_bias,
                bias_k,
                bias_v,
                add_zero_attn,
                dropout_p,
                out_proj_weight,
                out_proj_bias,
                training=training,
                key_padding_mask=key_padding_mask,
                need_weights=need_weights,
                attn_mask=attn_mask,
                use_separate_proj_weight=use_separate_proj_weight,
                q_proj_weight=q_proj_weight,
                k_proj_weight=k_proj_weight,
                v_proj_weight=v_proj_weight,
                static_k=static_k,
                static_v=static_v,
            )
    tgt_len, bsz, embed_dim = query.size()
    assert embed_dim == embed_dim_to_check
    # allow MHA to have different sizes for the feature dimension
    assert key.size(0) == value.size(0) and key.size(1) == value.size(1)

    head_dim = embed_dim // num_heads
    v_head_dim = out_dim // num_heads
    assert head_dim * num_heads == embed_dim, "embed_dim must be divisible by num_heads"
    scaling = float(head_dim) ** -0.5

    q = query * scaling
    k = key
    v = value

    if attn_mask is not None:
        assert (
            attn_mask.dtype == torch.float32
            or attn_mask.dtype == torch.float64
            or attn_mask.dtype == torch.float16
            or attn_mask.dtype == torch.uint8
            or attn_mask.dtype == torch.bool
        ), (
            "Only float, byte, and bool types are supported for attn_mask, not {}".format(
                attn_mask.dtype
            )
        )
        if attn_mask.dtype == torch.uint8:
            warnings.warn(
                "Byte tensor for attn_mask in nn.MultiheadAttention is deprecated. Use bool tensor instead."
            )
            attn_mask = attn_mask.to(torch.bool)

        if attn_mask.dim() == 2:
            attn_mask = attn_mask.unsqueeze(0)
            if list(attn_mask.size()) != [1, query.size(0), key.size(0)]:
                raise RuntimeError("The size of the 2D attn_mask is not correct.")
        elif attn_mask.dim() == 3:
            if list(attn_mask.size()) != [bsz * num_heads, query.size(0), key.size(0)]:
                raise RuntimeError("The size of the 3D attn_mask is not correct.")
        else:
            raise RuntimeError(
                "attn_mask's dimension {} is not supported".format(attn_mask.dim())
            )
        # attn_mask's dim is 3 now.

    # convert ByteTensor key_padding_mask to bool
    if key_padding_mask is not None and key_padding_mask.dtype == torch.uint8:
        warnings.warn(
            "Byte tensor for key_padding_mask in nn.MultiheadAttention is deprecated. Use bool tensor instead."
        )
        key_padding_mask = key_padding_mask.to(torch.bool)

    if bias_k is not None and bias_v is not None:
        if static_k is None and static_v is None:
            k = torch.cat([k, bias_k.repeat(1, bsz, 1)])
            v = torch.cat([v, bias_v.repeat(1, bsz, 1)])
            if attn_mask is not None:
                attn_mask = pad(attn_mask, (0, 1))
            if key_padding_mask is not None:
                key_padding_mask = pad(key_padding_mask, (0, 1))
        else:
            assert static_k is None, "bias cannot be added to static key."
            assert static_v is None, "bias cannot be added to static value."
    else:
        assert bias_k is None
        assert bias_v is None

    q = q.contiguous().view(tgt_len, bsz * num_heads, head_dim).transpose(0, 1)
    if k is not None:
        k = k.contiguous().view(-1, bsz * num_heads, head_dim).transpose(0, 1)
    if v is not None:
        v = v.contiguous().view(-1, bsz * num_heads, v_head_dim).transpose(0, 1)

    if static_k is not None:
        assert static_k.size(0) == bsz * num_heads
        assert static_k.size(2) == head_dim
        k = static_k

    if static_v is not None:
        assert static_v.size(0) == bsz * num_heads
        assert static_v.size(2) == v_head_dim
        v = static_v

    src_len = k.size(1)

    if key_padding_mask is not None:
        assert key_padding_mask.size(0) == bsz
        assert key_padding_mask.size(1) == src_len

    if add_zero_attn:
        src_len += 1
        k = torch.cat(
            [
                k,
                torch.zeros(
                    (k.size(0), 1) + k.size()[2:], dtype=k.dtype, device=k.device
                ),
            ],
            dim=1,
        )
        v = torch.cat(
            [
                v,
                torch.zeros(
                    (v.size(0), 1) + v.size()[2:], dtype=v.dtype, device=v.device
                ),
            ],
            dim=1,
        )
        if attn_mask is not None:
            attn_mask = pad(attn_mask, (0, 1))
        if key_padding_mask is not None:
            key_padding_mask = pad(key_padding_mask, (0, 1))

    attn_output_weights = torch.bmm(q, k.transpose(1, 2))
    assert list(attn_output_weights.size()) == [bsz * num_heads, tgt_len, src_len]

    if attn_mask is not None:
        if attn_mask.dtype == torch.bool:
            attn_output_weights.masked_fill_(attn_mask, float("-inf"))
        else:
            attn_output_weights += attn_mask

    if key_padding_mask is not None:
        attn_output_weights = attn_output_weights.view(bsz, num_heads, tgt_len, src_len)
        attn_output_weights = attn_output_weights.masked_fill(
            key_padding_mask.unsqueeze(1).unsqueeze(2),
            float("-inf"),
        )
        attn_output_weights = attn_output_weights.view(
            bsz * num_heads, tgt_len, src_len
        )

    # attn_output_weights = softmax(
    #     attn_output_weights, dim=-1)
    attn_output_weights = softmax(
        attn_output_weights - attn_output_weights.max(dim=-1, keepdim=True)[0], dim=-1
    )
    attn_output_weights = dropout(attn_output_weights, p=dropout_p, training=training)

    attn_output = torch.bmm(attn_output_weights, v)
    assert list(attn_output.size()) == [bsz * num_heads, tgt_len, v_head_dim]
    attn_output = attn_output.transpose(0, 1).contiguous().view(tgt_len, bsz, out_dim)
    attn_output = linear(attn_output, out_proj_weight, out_proj_bias)

    if need_weights:
        # average attention weights over heads
        attn_output_weights = attn_output_weights.view(bsz, num_heads, tgt_len, src_len)
        return attn_output, attn_output_weights.sum(dim=1) / num_heads
    else:
        return attn_output, None

In [31]:
class MultiheadAttention(Module):
    r"""Allows the model to jointly attend to information
    from different representation subspaces.
    See reference: Attention Is All You Need
    .. math::
        \text{MultiHead}(Q, K, V) = \text{Concat}(head_1,\dots,head_h)W^O
        \text{where} head_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)
    Args:
        embed_dim: total dimension of the model.
        num_heads: parallel attention heads.
        dropout: a Dropout layer on attn_output_weights. Default: 0.0.
        bias: add bias as module parameter. Default: True.
        add_bias_kv: add bias to the key and value sequences at dim=0.
        add_zero_attn: add a new batch of zeros to the key and
                       value sequences at dim=1.
        kdim: total number of features in key. Default: None.
        vdim: total number of features in value. Default: None.
        Note: if kdim and vdim are None, they will be set to embed_dim such that
        query, key, and value have the same number of features.
    Examples::
        >>> multihead_attn = nn.MultiheadAttention(embed_dim, num_heads)
        >>> attn_output, attn_output_weights = multihead_attn(query, key, value)
    """

    bias_k: Optional[torch.Tensor]
    bias_v: Optional[torch.Tensor]

    def __init__(
        self,
        embed_dim,
        num_heads,
        dropout=0.0,
        bias=True,
        add_bias_kv=False,
        add_zero_attn=False,
        kdim=None,
        vdim=None,
    ):
        super(MultiheadAttention, self).__init__()
        self.embed_dim = embed_dim
        self.kdim = kdim if kdim is not None else embed_dim
        self.vdim = vdim if vdim is not None else embed_dim
        self._qkv_same_embed_dim = self.kdim == embed_dim and self.vdim == embed_dim

        self.num_heads = num_heads
        self.dropout = dropout
        self.head_dim = embed_dim // num_heads
        assert self.head_dim * num_heads == self.embed_dim, (
            "embed_dim must be divisible by num_heads"
        )

        vdim = vdim if vdim is not None else embed_dim
        self.out_proj = Linear(vdim, vdim)

        self.in_proj_bias = None
        self.in_proj_weight = None
        self.bias_k = self.bias_v = None
        self.q_proj_weight = None
        self.k_proj_weight = None
        self.v_proj_weight = None

        self.add_zero_attn = add_zero_attn

        self._reset_parameters()

    def _reset_parameters(self):
        constant_(self.out_proj.bias, 0.0)

    def __setstate__(self, state):
        # Support loading old MultiheadAttention checkpoints generated by v1.1.0
        if "_qkv_same_embed_dim" not in state:
            state["_qkv_same_embed_dim"] = True

        super(MultiheadAttention, self).__setstate__(state)

    def forward(
        self,
        query,
        key,
        value,
        key_padding_mask=None,
        need_weights=True,
        attn_mask=None,
    ):
        # type: (Tensor, Tensor, Tensor, Optional[Tensor], bool, Optional[Tensor]) -> Tuple[Tensor, Optional[Tensor]]
        if not self._qkv_same_embed_dim:
            return multi_head_attention_forward(
                query,
                key,
                value,
                self.embed_dim,
                self.num_heads,
                self.in_proj_weight,
                self.in_proj_bias,
                self.bias_k,
                self.bias_v,
                self.add_zero_attn,
                self.dropout,
                self.out_proj.weight,
                self.out_proj.bias,
                training=self.training,
                key_padding_mask=key_padding_mask,
                need_weights=need_weights,
                attn_mask=attn_mask,
                use_separate_proj_weight=True,
                q_proj_weight=self.q_proj_weight,
                k_proj_weight=self.k_proj_weight,
                v_proj_weight=self.v_proj_weight,
                out_dim=self.vdim,
            )
        else:
            return multi_head_attention_forward(
                query,
                key,
                value,
                self.embed_dim,
                self.num_heads,
                self.in_proj_weight,
                self.in_proj_bias,
                self.bias_k,
                self.bias_v,
                self.add_zero_attn,
                self.dropout,
                self.out_proj.weight,
                self.out_proj.bias,
                training=self.training,
                key_padding_mask=key_padding_mask,
                need_weights=need_weights,
                attn_mask=attn_mask,
                out_dim=self.vdim,
            )

In [154]:
import torch
import torch.nn as nn

class CrossModalCoAttention(nn.Module):
    def __init__(self, d_model=512, nhead=8, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead

        # CRITICAL: Added batch_first=True to match your (32, L, 256) inputs
        self.audio_to_text = nn.MultiheadAttention(
            d_model, nhead, dropout=dropout, batch_first=True
        )
        self.text_to_audio = nn.MultiheadAttention(
            d_model, nhead, dropout=dropout, batch_first=True
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # Standard Transformer FFN uses separate norms for each modality
        self.ffn_a = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
        )
        self.ffn_t = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
        )
        
        self.norm_ffn_a = nn.LayerNorm(d_model)
        self.norm_ffn_t = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, audio_feat, text_feat, audio_mask=None, text_mask=None):
        """
        audio_feat: (B, L_a, D) -> (32, 61, 256)
        text_feat:  (B, L_t, D) -> (32, 25, 256)
        audio_mask: (B, L_a) - 1 for valid, 0 for pad
        text_mask:  (B, L_t) - 1 for valid, 0 for pad
        """
        
        # 1. Audio attends to Text
        # Note: key_padding_mask needs to be (B, L). 
        # If your mask is 1 for data and 0 for pad, we invert it (mask == 0)
        a_mask = (audio_mask == 0) if audio_mask is not None else None
        t_mask = (text_mask == 0) if text_mask is not None else None

        attn_a2t, _ = self.audio_to_text(
            query=audio_feat,
            key=text_feat,
            value=text_feat,
            key_padding_mask=t_mask
        )
        audio_feat = self.norm1(audio_feat + self.dropout(attn_a2t))

        # 2. Text attends to Audio
        attn_t2a, _ = self.text_to_audio(
            query=text_feat,
            key=audio_feat,
            value=audio_feat,
            key_padding_mask=a_mask
        )
        text_feat = self.norm2(text_feat + self.dropout(attn_t2a))

        # 3. FFN layers (Residual + Norm)
        audio_feat = self.norm_ffn_a(audio_feat + self.dropout(self.ffn_a(audio_feat)))
        text_feat = self.norm_ffn_t(text_feat + self.dropout(self.ffn_t(text_feat)))

        return audio_feat, text_feat


class CrossModalFusionBlock(nn.Module):
    def __init__(self, d_model=512, nhead=8, num_layers=2, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList(
            [CrossModalCoAttention(d_model, nhead, dropout) for _ in range(num_layers)]
        )

        # This layer is necessary if you want the output to remain d_model 
        # after concatenating two d_model tensors.
        self.projection = nn.Linear(d_model, d_model) 
        self.norm = nn.LayerNorm(d_model)

    def forward(self, audio_feat, text_feat, audio_mask=None, text_mask=None):
        # 1. Cross-attend
        for layer in self.layers:
            audio_feat, text_feat = layer(audio_feat, text_feat, audio_mask, text_mask)

        # 2. Concatenate along sequence dimension: (B, L_a + L_t, D)
        # For your inputs: (32, 61+25, 256) -> (32, 86, 256)
        fused = torch.cat([audio_feat, text_feat], dim=1) 
        
        # 3. (Optional) Project back to d_model if the next layer expects it
        # fused = self.norm(self.projection(fused))

        return fused

In [33]:
a_feat_temp = torch.Tensor(train_sample.get("model_inputs")["audio_feat"])
q_feat_temp = torch.Tensor(train_sample.get("model_inputs")["query_feat"])

In [34]:
base_pos_enc = PositionEmbeddingSine(256, normalize=True)

txt_pos_embed = TrainablePositionalEncoding(
    max_position_embeddings=32,
    hidden_size=256,
    dropout=0.5,
)

In [35]:
rotary_pos_enc = RotaryEmbedding(dim=256)
query_v = rotary_pos_enc(q_feat_temp)

In [36]:
query_v

(tensor([[[[ 1.0000,  1.0000,  1.0000,  ...,  1.0000,  1.0000,  1.0000],
           [ 0.5403,  0.5974,  0.6479,  ...,  1.0000,  1.0000,  1.0000],
           [-0.4161, -0.2863, -0.1604,  ...,  1.0000,  1.0000,  1.0000],
           ...,
           [ 0.0044, -0.6884, -0.9949,  ...,  1.0000,  1.0000,  1.0000],
           [ 0.8439,  0.1705, -0.5679,  ...,  1.0000,  1.0000,  1.0000],
           [ 0.9074,  0.8921,  0.2590,  ...,  1.0000,  1.0000,  1.0000]]]]),
 tensor([[[[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
             0.0000e+00,  0.0000e+00],
           [ 8.4147e-01,  8.0196e-01,  7.6172e-01,  ...,  1.2409e-04,
             1.1548e-04,  1.0746e-04],
           [ 9.0930e-01,  9.5814e-01,  9.8705e-01,  ...,  2.4819e-04,
             2.3096e-04,  2.1492e-04],
           ...,
           [-9.9999e-01, -7.2533e-01, -1.0066e-01,  ...,  1.3650e-03,
             1.2703e-03,  1.1821e-03],
           [-5.3657e-01, -9.8537e-01, -8.2307e-01,  ...,  1.4891e-03,
             1.3857e

#### Encoder

In [137]:
class TransformerEncoderLayerThin(nn.Module):
    def __init__(
        self,
        d_model,
        nhead,
        dim_feedforward=2048,
        dropout=0.1,
        activation="relu",
        normalize_before=False,
    ):
        """Initializes the TransformerEncoderLayerThin.

        Args:
            d_model (int): Model dimension.
            nhead (int): Number of heads.
            dim_feedforward (int): Feedforward dimension.
            dropout (float): Dropout rate.
            activation (str): Activation function.
            normalize_before (bool): Whether to normalize before.
        """
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        # Implementation of Feedforward model
        # self.linear1 = nn.Linear(d_model, dim_feedforward)
        # self.dropout = nn.Dropout(dropout)
        # self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.linear = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

        # self.activation = _get_activation_fn(activation)
        self.normalize_before = normalize_before

    def with_pos_embed(self, tensor, pos: Optional[Tensor]):
        """Adds positional embedding to the tensor.

        Args:
            tensor (torch.Tensor): Input tensor.
            pos (Optional[Tensor]): Positional embedding.

        Returns:
            torch.Tensor: Tensor with positional embedding.
        """
        return tensor if pos is None else tensor + pos

    def forward_post(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
    ):
        q = k = self.with_pos_embed(src, pos)
        src2 = self.self_attn(
            q, k, value=src, attn_mask=src_mask, key_padding_mask=src_key_padding_mask
        )[0]
        src2 = self.linear(src2)
        src = src + self.dropout(src2)
        src = self.norm(src)
        # src = src + self.dropout1(src2)
        # src = self.norm1(src)
        # src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        # src = src + self.dropout2(src2)
        # src = self.norm2(src)
        return src

    def forward_pre(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
    ):
        """not used"""
        src2 = self.norm1(src)
        q = k = self.with_pos_embed(src2, pos)
        src2 = self.self_attn(
            q, k, value=src2, attn_mask=src_mask, key_padding_mask=src_key_padding_mask
        )[0]
        src = src + self.dropout1(src2)
        src2 = self.norm2(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src2))))
        src = src + self.dropout2(src2)
        return src

    def forward(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
    ):
        if self.normalize_before:
            return self.forward_pre(src, src_mask, src_key_padding_mask, pos)
        return self.forward_post(src, src_mask, src_key_padding_mask, pos)


class T2V_TransformerEncoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        nhead,
        dim_feedforward=2048,
        dropout=0.1,
        activation="relu",
        normalize_before=False,
    ):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        # Implementation of Feedforward model
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        self.activation = _get_activation_fn(activation)
        self.normalize_before = normalize_before
        self.nhead = nhead

    def with_pos_embed(self, tensor, pos: Optional[Tensor]):
        return tensor if pos is None else tensor + pos

    def forward_post(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        audio_length=None,
    ):

        assert audio_length is not None

        # print('before src shape :', src.shape)
        pos_src = self.with_pos_embed(src, pos)
        global_token, q, k, v = (
            src[0].unsqueeze(0),
            pos_src[1 : audio_length + 1],
            pos_src[audio_length + 1 :],
            src[audio_length + 1 :],
        )

        # print(src_key_padding_mask.shape) # torch.Size([32, 102])
        # print(src_key_padding_mask[:, 1:76].permute(1,0).shape) # torch.Size([75, 32])
        # print(src_key_padding_mask[:, 76:].shape) # torch.Size([32, 26])

        qmask, kmask = (
            src_key_padding_mask[:, 1 : audio_length + 1].unsqueeze(2),
            src_key_padding_mask[:, audio_length + 1 :].unsqueeze(1),
        )
        attn_mask = (
            torch.matmul(qmask.float(), kmask.float()).bool().repeat(self.nhead, 1, 1)
        )
        # print(attn_mask.shape)
        # print(attn_mask[0][0])
        # print(q.shape) 75 32 256
        # print(k.shape) 26 32 256

        src2 = self.self_attn(
            q,
            k,
            value=v,
            attn_mask=attn_mask,
            key_padding_mask=src_key_padding_mask[:, audio_length + 1 :],
        )[0]
        src2 = src[1 : audio_length + 1] + self.dropout1(src2)
        src3 = self.norm1(src2)
        src3 = self.linear2(self.dropout(self.activation(self.linear1(src3))))
        src2 = src2 + self.dropout2(src3)
        src2 = self.norm2(src2)
        src2 = torch.cat([global_token, src2], dim=0)
        src = torch.cat([src2, src[audio_length + 1 :]])
        # print('after src shape :',src.shape)
        return src

    def forward_pre(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
    ):
        print("before src shape :", src.shape)
        src2 = self.norm1(src)
        pos_src = self.with_pos_embed(src2, pos)
        global_token, q, k, v = (
            src[0].unsqueeze(0),
            pos_src[1:76],
            pos_src[76:],
            src2[76:],
        )
        # print(q.shape) # 100 32 256

        src2 = self.self_attn(
            q,
            k,
            value=v,
            attn_mask=src_key_padding_mask[:, 1:76].permute(1, 0),
            key_padding_mask=src_key_padding_mask[:, 76:],
        )[0]
        src2 = src[1:76] + self.dropout1(src2)
        src3 = self.norm1(src2)
        src3 = self.linear2(self.dropout(self.activation(self.linear1(src3))))
        src2 = src2 + self.dropout2(src3)
        src2 = self.norm2(src2)
        src2 = torch.cat([global_token, src2], dim=0)
        src = torch.cat([src2, src[76:]])
        print("after src shape :", src.shape)
        return src

    def forward(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        **kwargs,
    ):
        if self.normalize_before:
            return self.forward_pre(src, src_mask, src_key_padding_mask, pos)
        # For tvsum, add kwargs
        return self.forward_post(src, src_mask, src_key_padding_mask, pos, **kwargs)


class TransformerEncoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        nhead,
        dim_feedforward=2048,
        dropout=0.1,
        activation="relu",
        normalize_before=False,
    ):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        # Implementation of Feedforward model
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        self.activation = _get_activation_fn(activation)
        self.normalize_before = normalize_before

    def with_pos_embed(self, tensor, pos: Optional[Tensor]):
        return tensor if pos is None else tensor + pos

    def forward_post(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
    ):
        q = k = self.with_pos_embed(src, pos)
        src2 = self.self_attn(
            q, k, value=src, attn_mask=src_mask, key_padding_mask=src_key_padding_mask
        )[0]
        src = src + self.dropout1(src2)
        src = self.norm1(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = src + self.dropout2(src2)
        src = self.norm2(src)
        return src

    def forward_pre(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
    ):
        src2 = self.norm1(src)
        q = k = self.with_pos_embed(src2, pos)
        src2 = self.self_attn(
            q, k, value=src2, attn_mask=src_mask, key_padding_mask=src_key_padding_mask
        )[0]
        src = src + self.dropout1(src2)
        src2 = self.norm2(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src2))))
        src = src + self.dropout2(src2)
        return src

    def forward(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
    ):
        if self.normalize_before:
            return self.forward_pre(src, src_mask, src_key_padding_mask, pos)
        return self.forward_post(src, src_mask, src_key_padding_mask, pos)

In [38]:
class TransformerEncoder(nn.Module):
    def __init__(self, encoder_layer, num_layers, norm=None, return_intermediate=False):
        """Initializes the TransformerEncoder.

        Args:
            encoder_layer: Encoder layer module.
            num_layers (int): Number of layers.
            norm: Normalization layer.
            return_intermediate (bool): Whether to return intermediate outputs.
        """
        super().__init__()
        self.layers = _get_clones(encoder_layer, num_layers)
        self.num_layers = num_layers
        self.norm = norm
        self.return_intermediate = return_intermediate

    def forward(
        self,
        src,
        mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        **kwargs,
    ):
        """Forward pass through the TransformerEncoder.

        Args:
            src (torch.Tensor): Source tensor.
            mask (Optional[Tensor]): Mask tensor.
            src_key_padding_mask (Optional[Tensor]): Key padding mask.
            pos (Optional[Tensor]): Position tensor.
            **kwargs: Additional arguments.

        Returns:
            torch.Tensor: Output tensor.
        """
        output = src

        intermediate = []

        for layer in self.layers:
            output = layer(
                output,
                src_mask=mask,
                src_key_padding_mask=src_key_padding_mask,
                pos=pos,
                **kwargs,
            )
            if self.return_intermediate:
                intermediate.append(output)

        if self.norm is not None:
            output = self.norm(output)

        if self.return_intermediate:
            return torch.stack(intermediate)

        return output

#### Decoder

In [39]:
class TransformerDecoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        nhead,
        dim_feedforward=2048,
        dropout=0.1,
        activation="relu",
        normalize_before=False,
        keep_query_pos=False,
        rm_self_attn_decoder=False,
    ):
        super().__init__()
        # Decoder Self-Attention
        if not rm_self_attn_decoder:
            self.sa_qcontent_proj = nn.Linear(d_model, d_model)
            self.sa_qpos_proj = nn.Linear(d_model, d_model)
            self.sa_kcontent_proj = nn.Linear(d_model, d_model)
            self.sa_kpos_proj = nn.Linear(d_model, d_model)
            self.sa_v_proj = nn.Linear(d_model, d_model)
            self.self_attn = MultiheadAttention(
                d_model, nhead, dropout=dropout, vdim=d_model
            )

            self.norm1 = nn.LayerNorm(d_model)
            self.dropout1 = nn.Dropout(dropout)

        # Decoder Cross-Attention
        self.ca_qcontent_proj = nn.Linear(d_model, d_model)
        self.ca_qpos_proj = nn.Linear(d_model, d_model)
        self.ca_kcontent_proj = nn.Linear(d_model, d_model)
        self.ca_kpos_proj = nn.Linear(d_model, d_model)
        self.ca_v_proj = nn.Linear(d_model, d_model)
        self.ca_qpos_sine_proj = nn.Linear(d_model, d_model)
        self.cross_attn = MultiheadAttention(
            d_model * 2, nhead, dropout=dropout, vdim=d_model
        )

        self.nhead = nhead
        self.rm_self_attn_decoder = rm_self_attn_decoder

        # Implementation of Feedforward model
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

        self.activation = _get_activation_fn(activation)
        self.normalize_before = normalize_before
        self.keep_query_pos = keep_query_pos

    def with_pos_embed(self, tensor, pos: Optional[Tensor]):
        return tensor if pos is None else tensor + pos

    def forward(
        self,
        tgt,
        memory,
        tgt_mask: Optional[Tensor] = None,
        memory_mask: Optional[Tensor] = None,
        tgt_key_padding_mask: Optional[Tensor] = None,
        memory_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        query_pos: Optional[Tensor] = None,
        query_sine_embed=None,
        is_first=False,
    ):

        # ========== Begin of Self-Attention =============
        if not self.rm_self_attn_decoder:
            # Apply projections here
            # shape: num_queries x batch_size x 256
            q_content = self.sa_qcontent_proj(
                tgt
            )  # target is the input of the first decoder layer. zero by default.
            q_pos = self.sa_qpos_proj(query_pos)
            k_content = self.sa_kcontent_proj(tgt)
            k_pos = self.sa_kpos_proj(query_pos)
            v = self.sa_v_proj(tgt)

            num_queries, bs, n_model = q_content.shape
            hw, _, _ = k_content.shape

            q = q_content + q_pos
            k = k_content + k_pos

            tgt2 = self.self_attn(
                q, k, value=v, attn_mask=tgt_mask, key_padding_mask=tgt_key_padding_mask
            )[0]
            # ========== End of Self-Attention =============

            tgt = tgt + self.dropout1(tgt2)
            tgt = self.norm1(tgt)

        # ========== Begin of Cross-Attention =============
        # Apply projections here
        # shape: num_queries x batch_size x 256
        q_content = self.ca_qcontent_proj(tgt)
        k_content = self.ca_kcontent_proj(memory)
        v = self.ca_v_proj(memory)

        num_queries, bs, n_model = q_content.shape
        hw, _, _ = k_content.shape

        k_pos = self.ca_kpos_proj(pos)

        # For the first decoder layer, we concatenate the positional embedding predicted from
        # the object query (the positional embedding) into the original query (key) in DETR.
        if is_first or self.keep_query_pos:
            q_pos = self.ca_qpos_proj(query_pos)
            q = q_content + q_pos
            k = k_content + k_pos
        else:
            q = q_content
            k = k_content

        q = q.view(num_queries, bs, self.nhead, n_model // self.nhead)
        query_sine_embed = self.ca_qpos_sine_proj(query_sine_embed)
        query_sine_embed = query_sine_embed.view(
            num_queries, bs, self.nhead, n_model // self.nhead
        )
        q = torch.cat([q, query_sine_embed], dim=3).view(num_queries, bs, n_model * 2)
        k = k.view(hw, bs, self.nhead, n_model // self.nhead)
        k_pos = k_pos.view(hw, bs, self.nhead, n_model // self.nhead)
        k = torch.cat([k, k_pos], dim=3).view(hw, bs, n_model * 2)

        tgt2 = self.cross_attn(
            query=q,
            key=k,
            value=v,
            attn_mask=memory_mask,
            key_padding_mask=memory_key_padding_mask,
        )[0]
        # ========== End of Cross-Attention =============

        tgt = tgt + self.dropout2(tgt2)
        tgt = self.norm2(tgt)
        tgt2 = self.linear2(self.dropout(self.activation(self.linear1(tgt))))
        tgt = tgt + self.dropout3(tgt2)
        tgt = self.norm3(tgt)
        return tgt


class TransformerDecoderLayerThin(nn.Module):
    """removed intermediate layer"""

    def __init__(
        self,
        d_model,
        nhead,
        dim_feedforward=2048,
        dropout=0.1,
        activation="relu",
        normalize_before=False,
    ):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.multihead_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        # Implementation of Feedforward model
        self.linear1 = nn.Linear(d_model, d_model)
        # self.linear1 = nn.Linear(d_model, dim_feedforward)
        # self.dropout = nn.Dropout(dropout)
        # self.linear2 = nn.Linear(dim_feedforward, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        # self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        # self.dropout3 = nn.Dropout(dropout)

        # self.activation = _get_activation_fn(activation)
        self.normalize_before = normalize_before

    def with_pos_embed(self, tensor, pos: Optional[Tensor]):
        return tensor if pos is None else tensor + pos

    def forward_post(
        self,
        tgt,
        memory,
        tgt_mask: Optional[Tensor] = None,
        memory_mask: Optional[Tensor] = None,
        tgt_key_padding_mask: Optional[Tensor] = None,
        memory_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        query_pos: Optional[Tensor] = None,
    ):
        q = k = self.with_pos_embed(tgt, query_pos)
        tgt2 = self.self_attn(
            q, k, value=tgt, attn_mask=tgt_mask, key_padding_mask=tgt_key_padding_mask
        )[0]
        tgt = tgt + self.dropout1(tgt2)
        tgt = self.norm1(tgt)
        tgt2 = self.multihead_attn(
            query=self.with_pos_embed(tgt, query_pos),
            key=self.with_pos_embed(memory, pos),
            value=memory,
            attn_mask=memory_mask,
            key_padding_mask=memory_key_padding_mask,
        )[0]
        tgt2 = self.linear1(tgt2)
        tgt = tgt + self.dropout2(tgt2)
        tgt = self.norm2(tgt)
        # tgt = tgt + self.dropout2(tgt2)
        # tgt = self.norm2(tgt)
        # tgt2 = self.linear2(self.dropout(self.activation(self.linear1(tgt))))
        # tgt = tgt + self.dropout3(tgt2)
        # tgt = self.norm3(tgt)
        return tgt

    def forward_pre(
        self,
        tgt,
        memory,
        tgt_mask: Optional[Tensor] = None,
        memory_mask: Optional[Tensor] = None,
        tgt_key_padding_mask: Optional[Tensor] = None,
        memory_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        query_pos: Optional[Tensor] = None,
    ):
        tgt2 = self.norm1(tgt)
        q = k = self.with_pos_embed(tgt2, query_pos)
        tgt2 = self.self_attn(
            q, k, value=tgt2, attn_mask=tgt_mask, key_padding_mask=tgt_key_padding_mask
        )[0]
        tgt = tgt + self.dropout1(tgt2)
        tgt2 = self.norm2(tgt)
        tgt2 = self.multihead_attn(
            query=self.with_pos_embed(tgt2, query_pos),
            key=self.with_pos_embed(memory, pos),
            value=memory,
            attn_mask=memory_mask,
            key_padding_mask=memory_key_padding_mask,
        )[0]
        tgt = tgt + self.dropout2(tgt2)
        tgt2 = self.norm3(tgt)
        tgt2 = self.linear2(self.dropout(self.activation(self.linear1(tgt2))))
        tgt = tgt + self.dropout3(tgt2)
        return tgt

    def forward(
        self,
        tgt,
        memory,
        tgt_mask: Optional[Tensor] = None,
        memory_mask: Optional[Tensor] = None,
        tgt_key_padding_mask: Optional[Tensor] = None,
        memory_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        query_pos: Optional[Tensor] = None,
    ):
        if self.normalize_before:
            return self.forward_pre(
                tgt,
                memory,
                tgt_mask,
                memory_mask,
                tgt_key_padding_mask,
                memory_key_padding_mask,
                pos,
                query_pos,
            )
        return self.forward_post(
            tgt,
            memory,
            tgt_mask,
            memory_mask,
            tgt_key_padding_mask,
            memory_key_padding_mask,
            pos,
            query_pos,
        )

In [40]:
class TransformerDecoder(nn.Module):
    def __init__(
        self,
        decoder_layer,
        num_layers,
        norm=None,
        return_intermediate=False,
        d_model=256,
        query_dim=2,
        keep_query_pos=False,
        query_scale_type="cond_elewise",
        modulate_t_attn=False,
        bbox_embed_diff_each_layer=False,
    ):
        super().__init__()
        self.layers = _get_clones(decoder_layer, num_layers)
        self.num_layers = num_layers
        self.norm = norm
        self.return_intermediate = return_intermediate
        assert return_intermediate
        self.query_dim = query_dim

        assert query_scale_type in ["cond_elewise", "cond_scalar", "fix_elewise"]
        self.query_scale_type = query_scale_type
        if query_scale_type == "cond_elewise":
            self.query_scale = MLP(d_model, d_model, d_model, 2)
        elif query_scale_type == "cond_scalar":
            self.query_scale = MLP(d_model, d_model, 1, 2)
        elif query_scale_type == "fix_elewise":
            self.query_scale = nn.Embedding(num_layers, d_model)
        else:
            raise NotImplementedError(
                "Unknown query_scale_type: {}".format(query_scale_type)
            )

        self.ref_point_head = MLP(d_model, d_model, d_model, 2)

        # self.bbox_embed = None
        # for DAB-deter
        if bbox_embed_diff_each_layer:
            self.bbox_embed = nn.ModuleList(
                [MLP(d_model, d_model, 2, 3) for i in range(num_layers)]
            )
        else:
            self.bbox_embed = MLP(d_model, d_model, 2, 3)
        # init bbox_embed
        if bbox_embed_diff_each_layer:
            for bbox_embed in self.bbox_embed:
                nn.init.constant_(bbox_embed.layers[-1].weight.data, 0)
                nn.init.constant_(bbox_embed.layers[-1].bias.data, 0)
        else:
            nn.init.constant_(self.bbox_embed.layers[-1].weight.data, 0)
            nn.init.constant_(self.bbox_embed.layers[-1].bias.data, 0)
        self.d_model = d_model
        self.modulate_t_attn = modulate_t_attn
        self.bbox_embed_diff_each_layer = bbox_embed_diff_each_layer

        if modulate_t_attn:
            self.ref_anchor_head = MLP(d_model, d_model, 1, 2)

        if not keep_query_pos:
            for layer_id in range(num_layers - 1):
                self.layers[layer_id + 1].ca_qpos_proj = None

    def forward(
        self,
        tgt,
        memory,
        tgt_mask: Optional[Tensor] = None,
        memory_mask: Optional[Tensor] = None,
        tgt_key_padding_mask: Optional[Tensor] = None,
        memory_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        refpoints_unsigmoid: Optional[Tensor] = None,  # num_queries, bs, 2
    ):
        output = tgt

        intermediate = []
        reference_points = refpoints_unsigmoid.sigmoid()
        ref_points = [reference_points]

        for layer_id, layer in enumerate(self.layers):
            obj_center = reference_points[..., : self.query_dim]
            # get sine embedding for the query vector
            query_sine_embed = gen_sineembed_for_position(obj_center)
            # print('line230', query_sine_embed.shape)
            query_pos = self.ref_point_head(query_sine_embed)
            # print('line232',query_sine_embed.shape)
            # For the first decoder layer, we do not apply transformation over p_s
            if self.query_scale_type != "fix_elewise":
                if layer_id == 0:
                    pos_transformation = 1
                else:
                    pos_transformation = self.query_scale(output)
            else:
                pos_transformation = self.query_scale.weight[layer_id]

            # apply transformation
            # print(query_sine_embed.shape) # 10 32 512
            query_sine_embed = query_sine_embed * pos_transformation

            # modulated HW attentions
            if self.modulate_t_attn:
                reft_cond = self.ref_anchor_head(output).sigmoid()  # nq, bs, 1
                # print(reft_cond.shape, reft_cond[..., 0].shape) # 10 32 1, 10 32
                # print(obj_center.shape, obj_center[..., 1].shape) # 10 32 2, 10 32
                # print(query_sine_embed.shape) # 10 32 256

                query_sine_embed *= (reft_cond[..., 0] / obj_center[..., 1]).unsqueeze(
                    -1
                )

            output = layer(
                output,
                memory,
                tgt_mask=tgt_mask,
                memory_mask=memory_mask,
                tgt_key_padding_mask=tgt_key_padding_mask,
                memory_key_padding_mask=memory_key_padding_mask,
                pos=pos,
                query_pos=query_pos,
                query_sine_embed=query_sine_embed,
                is_first=(layer_id == 0),
            )

            # iter update
            if self.bbox_embed is not None:
                if self.bbox_embed_diff_each_layer:
                    tmp = self.bbox_embed[layer_id](output)
                else:
                    tmp = self.bbox_embed(output)
                tmp[..., : self.query_dim] += inverse_sigmoid(reference_points)
                new_reference_points = tmp[..., : self.query_dim].sigmoid()
                if layer_id != self.num_layers - 1:
                    ref_points.append(new_reference_points)
                reference_points = new_reference_points.detach()

            if self.return_intermediate:
                intermediate.append(self.norm(output))

        if self.norm is not None:
            output = self.norm(output)
            if self.return_intermediate:
                intermediate.pop()
                intermediate.append(output)

        if self.return_intermediate:
            if self.bbox_embed is not None:
                return [
                    torch.stack(intermediate).transpose(1, 2),
                    torch.stack(ref_points).transpose(1, 2),
                ]
            else:
                return [
                    torch.stack(intermediate).transpose(1, 2),
                    reference_points.unsqueeze(0).transpose(1, 2),
                ]

        return output.unsqueeze(0)

In [41]:
class Transformer(nn.Module):
    def __init__(
        self,
        d_model=512,
        nhead=8,
        num_queries=2,
        num_encoder_layers=6,
        num_decoder_layers=6,
        dim_feedforward=2048,
        dropout=0.1,
        activation="relu",
        normalize_before=False,
        return_intermediate_dec=False,
        query_dim=2,
        keep_query_pos=False,
        query_scale_type="cond_elewise",
        num_patterns=0,
        modulate_t_attn=True,
        bbox_embed_diff_each_layer=False,
    ):
        """Initializes the Transformer model.

        Args:
            d_model (int): Model dimension.
            nhead (int): Number of attention heads.
            num_queries (int): Number of queries.
            num_encoder_layers (int): Number of encoder layers.
            num_decoder_layers (int): Number of decoder layers.
            dim_feedforward (int): Feedforward dimension.
            dropout (float): Dropout rate.
            activation (str): Activation function.
            normalize_before (bool): Whether to normalize before.
            return_intermediate_dec (bool): Whether to return intermediate decoder outputs.
            query_dim (int): Query dimension.
            keep_query_pos (bool): Whether to keep query position.
            query_scale_type (str): Query scale type.
            num_patterns (int): Number of patterns.
            modulate_t_attn (bool): Whether to modulate temporal attention.
            bbox_embed_diff_each_layer (bool): Whether bbox embed differs each layer.
        """
        super().__init__()

        t2v_encoder_layer = T2V_TransformerEncoderLayer(
            d_model, nhead, dim_feedforward, dropout, activation, normalize_before
        )
        encoder_norm = nn.LayerNorm(d_model) if normalize_before else None
        self.t2v_encoder = TransformerEncoder(
            t2v_encoder_layer, num_encoder_layers, encoder_norm
        )

        # TransformerEncoderLayerThin
        encoder_layer = TransformerEncoderLayer(
            d_model, nhead, dim_feedforward, dropout, activation, normalize_before
        )
        encoder_norm = nn.LayerNorm(d_model) if normalize_before else None
        self.encoder = TransformerEncoder(
            encoder_layer, num_encoder_layers, encoder_norm
        )

        # TransformerDecoderLayerThin
        decoder_layer = TransformerDecoderLayer(
            d_model,
            nhead,
            dim_feedforward,
            dropout,
            activation,
            normalize_before,
            keep_query_pos=keep_query_pos,
        )
        decoder_norm = nn.LayerNorm(d_model)
        self.decoder = TransformerDecoder(
            decoder_layer,
            num_decoder_layers,
            decoder_norm,
            return_intermediate=return_intermediate_dec,
            d_model=d_model,
            query_dim=query_dim,
            keep_query_pos=keep_query_pos,
            query_scale_type=query_scale_type,
            modulate_t_attn=modulate_t_attn,
            bbox_embed_diff_each_layer=bbox_embed_diff_each_layer,
        )

        self._reset_parameters()

        self.d_model = d_model
        self.nhead = nhead
        self.dec_layers = num_decoder_layers
        self.num_queries = num_queries
        self.num_patterns = num_patterns

    def _reset_parameters(self):
        """Resets the parameters of the model."""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, src, mask, query_embed, pos_embed, audio_length):
        """
        Args:
            src: (batch_size, L, d)
            mask: (batch_size, L)
            query_embed: (#queries, d)
            pos_embed: (batch_size, L, d) the same as src

        Returns:

        """
        # flatten NxCxHxW to HWxNxC
        bs, l, d = src.shape
        src = src.permute(1, 0, 2)  # (L, batch_size, d)
        pos_embed = pos_embed.permute(1, 0, 2)  # (L, batch_size, d)
        refpoint_embed = query_embed.unsqueeze(1).repeat(
            1, bs, 1
        )  # (#queries, batch_size, d)

        src = self.t2v_encoder(
            src, src_key_padding_mask=mask, pos=pos_embed, audio_length=audio_length
        )  # (L, batch_size, d)
        # print('after encoder : ',src.shape)
        src = src[: audio_length + 1]
        mask = mask[:, : audio_length + 1]
        pos_embed = pos_embed[: audio_length + 1]

        memory = self.encoder(
            src, src_key_padding_mask=mask, pos=pos_embed
        )  # (L, batch_size, d)
        memory_global, memory_local = memory[0], memory[1:]
        mask_local = mask[:, 1:]
        pos_embed_local = pos_embed[1:]

        tgt = torch.zeros(refpoint_embed.shape[0], bs, d, device=src.device)
        hs, references = self.decoder(
            tgt,
            memory_local,
            memory_key_padding_mask=mask_local,
            pos=pos_embed_local,
            refpoints_unsigmoid=refpoint_embed,
        )  # (#layers, #queries, batch_size, d)
        # hs = hs.transpose(1, 2)  # (#layers, batch_size, #qeries, d)
        # memory = memory.permute(1, 2, 0)  # (batch_size, d, L)
        memory_local = memory_local.transpose(0, 1)  # (batch_size, L, d)
        return hs, references, memory_local, memory_global

In [42]:
def _get_activation_fn(activation):
    """Return an activation function given a string"""
    if activation == "relu":
        return F.relu
    if activation == "gelu":
        return F.gelu
    if activation == "glu":
        return F.glu
    if activation == "prelu":
        return nn.PReLU()
    if activation == "selu":
        return F.selu
    raise RuntimeError(f"activation should be relu/gelu, not {activation}.")


def _get_clones(module, N):
    """Creates N clones of the module.

    Args:
        module: Module to clone.
        N (int): Number of clones.

    Returns:
        nn.ModuleList: List of cloned modules.
    """
    return nn.ModuleList([copy.deepcopy(module) for i in range(N)])


def build_transformer(args):
    """Builds the transformer model.

    Args:
        args: Configuration arguments.

    Returns:
        Transformer: The transformer model.
    """
    return Transformer(
        d_model=args.hidden_dim,
        dropout=args.dropout,
        nhead=args.nheads,
        dim_feedforward=args.dim_feedforward,
        num_encoder_layers=args.enc_layers,
        num_decoder_layers=args.dec_layers,
        normalize_before=False,
        return_intermediate_dec=True,
        activation="prelu",
    )

#### DETR-based Arch


In [164]:
class QDDETR(nn.Module):
    def __init__(
        self,
        transformer,
        position_embed,
        txt_position_embed,
        aud_dim,
        txt_dim,
        num_queries,
        input_dropout,
        max_a_l,
        aux_loss=True,
        span_loss_type="l1",
        use_txt_pos=False,
        n_input_proj=2,
    ):
        """Initializes the model.
        Parameters:
            transformer: torch module of the transformer architecture. See transformer.py
            position_embed: torch module of the position_embedding, See position_encoding.py
            txt_position_embed: position_embedding for text
            txt_dim: int, text query input dimension
            num_queries: number of object queries, ie detection slot. This is the maximal number of objects
                         QD-DETR can detect in a single audio.
            aux_loss: True if auxiliary decoding losses (loss at each decoder layer) are to be used.
            max_a_l: int, maximum #clips in audio
            span_loss_type: str, one of [l1, ce]
                l1: (center-x, width) regression.
                ce: (st_idx, ed_idx) classification.
            # foreground_thd: float, intersection over prediction >= foreground_thd: labeled as foreground
            # background_thd: float, intersection over prediction <= background_thd: labeled background
        """
        super().__init__()
        self.num_queries = num_queries
        self.transformer = transformer
        self.position_embed = position_embed
        self.txt_position_embed = txt_position_embed
        hidden_dim = transformer.d_model
        self.span_loss_type = span_loss_type
        self.max_a_l = max_a_l
        span_pred_dim = 2
        self.span_embed = MLP(hidden_dim, hidden_dim, span_pred_dim, 3)
        self.class_embed = nn.Linear(hidden_dim, 2)  # 0: background, 1: foreground
        self.use_txt_pos = use_txt_pos
        self.n_input_proj = n_input_proj
        self.query_embed = nn.Embedding(num_queries, 2)
        relu_args = [True] * 3
        relu_args[n_input_proj - 1] = False

        self.input_txt_proj = nn.Sequential(
            *[
                LinearLayer(
                    txt_dim,
                    hidden_dim,
                    layer_norm=True,
                    dropout=input_dropout,
                    relu=relu_args[0],
                ),
                LinearLayer(
                    hidden_dim,
                    hidden_dim,
                    layer_norm=True,
                    dropout=input_dropout,
                    relu=relu_args[1],
                ),
                LinearLayer(
                    hidden_dim,
                    hidden_dim,
                    layer_norm=True,
                    dropout=input_dropout,
                    relu=relu_args[2],
                ),
            ][:n_input_proj]
        )
        self.input_aud_proj = nn.Sequential(
            *[
                LinearLayer(
                    aud_dim + 2,
                    hidden_dim,
                    layer_norm=True,
                    dropout=input_dropout,
                    relu=relu_args[0],
                ),  # add pos_embedding
                LinearLayer(
                    hidden_dim,
                    hidden_dim,
                    layer_norm=True,
                    dropout=input_dropout,
                    relu=relu_args[1],
                ),
                LinearLayer(
                    hidden_dim,
                    hidden_dim,
                    layer_norm=True,
                    dropout=input_dropout,
                    relu=relu_args[2],
                ),
            ][:n_input_proj]
        )
        self.aux_loss = aux_loss

        self.saliency_proj1 = nn.Linear(hidden_dim, hidden_dim)
        self.saliency_proj2 = nn.Linear(hidden_dim, hidden_dim)

        self.hidden_dim = hidden_dim
        self.global_rep_token = torch.nn.Parameter(torch.randn(hidden_dim))
        self.global_rep_pos = torch.nn.Parameter(torch.randn(hidden_dim))
        self.cross_modal_fusion = CrossModalFusionBlock(
            d_model=hidden_dim,
            nhead=self.transformer.nhead,
            num_layers=2,  # start with 1~3
            dropout=input_dropout,
        )

    def forward(self, src_txt, src_txt_mask, src_aud, src_aud_mask):
        """The forward expects two tensors:
           - src_txt: [batch_size, L_txt, D_txt]
           - src_txt_mask: [batch_size, L_txt], containing 0 on padded pixels,
                will convert to 1 as padding later for transformer
           - src_aud: [batch_size, L_aud, D_aud]
           - src_aud_mask: [batch_size, L_aud], containing 0 on padded pixels,
                will convert to 1 as padding later for transformer

        It returns a dict with the following elements:
           - "pred_spans": The normalized boxes coordinates for all queries, represented as
                           (center_x, width). These values are normalized in [0, 1],
                           relative to the size of each individual image (disregarding possible padding).
                           See PostProcess for information on how to retrieve the unnormalized bounding box.
           - "aux_outputs": Optional, only returned when auxilary losses are activated. It is a list of
                            dictionnaries containing the two above keys for each decoder layer.
        """
        src_aud = self.input_aud_proj(src_aud)
        src_txt = self.input_txt_proj(src_txt)

        # === CROSS-MODAL FUSION ===
        fused_feat = self.cross_modal_fusion(
            src_aud,
            src_txt,
            audio_mask=src_aud_mask.bool(),
            text_mask=src_txt_mask.bool(),
        )

        # Now use fused_feat instead of concatenation
        src = fused_feat
        mask = torch.cat([src_aud_mask, src_txt_mask], dim=1).bool()

        # Position embedding
        pos_aud = self.position_embed(src_aud, src_aud_mask)
        pos_txt = (
            self.txt_position_embed(src_txt)
            if self.use_txt_pos
            else torch.zeros_like(src_txt)
        )
        pos = torch.cat([pos_aud, pos_txt], dim=1)
        # (#layers, bsz, #queries, d), (bsz, L_aud+L_txt, d)

        # for global token
        mask_ = torch.tensor([[True]]).to(mask.device).repeat(mask.shape[0], 1)
        mask = torch.cat([mask_, mask], dim=1)
        src_ = self.global_rep_token.reshape([1, 1, self.hidden_dim]).repeat(
            src.shape[0], 1, 1
        )
        src = torch.cat([src_, src], dim=1)
        pos_ = self.global_rep_pos.reshape([1, 1, self.hidden_dim]).repeat(
            pos.shape[0], 1, 1
        )
        pos = torch.cat([pos_, pos], dim=1)

        audio_length = src_aud.shape[1]

        hs, reference, memory, memory_global = self.transformer(
            src, ~mask, self.query_embed.weight, pos, audio_length
        )
        outputs_class = self.class_embed(
            hs
        )  # (#layers, batch_size, #queries, #classes)
        reference_before_sigmoid = inverse_sigmoid(reference)
        tmp = self.span_embed(hs)
        outputs_coord = tmp + reference_before_sigmoid
        if self.span_loss_type == "l1":
            outputs_coord = outputs_coord.sigmoid()
        out = {"pred_logits": outputs_class[-1], "pred_spans": outputs_coord[-1]}

        txt_mem = memory[:, src_aud.shape[1] :]  # (bsz, L_txt, d)
        aud_mem = memory[:, : src_aud.shape[1]]  # (bsz, L_aud, d)

        ### Neg Pairs ###
        src_txt_neg = torch.cat([src_txt[1:], src_txt[0:1]], dim=0)
        src_txt_mask_neg = torch.cat([src_txt_mask[1:], src_txt_mask[0:1]], dim=0)
        src_neg = torch.cat([src_aud, src_txt_neg], dim=1)
        mask_neg = torch.cat([src_aud_mask, src_txt_mask_neg], dim=1).bool()

        mask_neg = torch.cat([mask_, mask_neg], dim=1)
        src_neg = torch.cat([src_, src_neg], dim=1)
        pos_neg = pos.clone()  # since it does not use actual content

        _, _, memory_neg, memory_global_neg = self.transformer(
            src_neg, ~mask_neg, self.query_embed.weight, pos_neg, audio_length
        )
        aud_mem_neg = memory_neg[:, : src_aud.shape[1]]

        out["saliency_scores"] = torch.sum(
            self.saliency_proj1(aud_mem)
            * self.saliency_proj2(memory_global).unsqueeze(1),
            dim=-1,
        ) / np.sqrt(self.hidden_dim)
        out["saliency_scores_neg"] = torch.sum(
            self.saliency_proj1(aud_mem_neg)
            * self.saliency_proj2(memory_global_neg).unsqueeze(1),
            dim=-1,
        ) / np.sqrt(self.hidden_dim)
        out["audio_mask"] = src_aud_mask
        if self.aux_loss:
            out["aux_outputs"] = [
                {"pred_logits": a, "pred_spans": b}
                for a, b in zip(outputs_class[:-1], outputs_coord[:-1])
            ]
        return out

In [44]:
class PostProcessorDETR:
    def __init__(
        self,
        clip_length=2,
        min_ts_val=0,
        max_ts_val=150,
        min_w_l=2,
        max_w_l=70,
        move_window_method="center",
        process_func_names=("clip_window_l", "clip_ts", "round_multiple"),
    ):
        self.clip_length = clip_length
        self.min_ts_val = min_ts_val
        self.max_ts_val = max_ts_val
        self.min_w_l = min_w_l
        self.max_w_l = max_w_l
        self.move_window_method = move_window_method
        self.process_func_names = process_func_names
        self.name2func = dict(
            clip_ts=self.clip_min_max_timestamps,
            round_multiple=self.round_to_multiple_clip_lengths,
        )

    def __call__(self, lines):
        processed_lines = []
        for line in tqdm(
            lines, desc=f"convert to multiples of clip_length={self.clip_length}"
        ):
            windows_and_scores = torch.tensor(line["pred_relevant_windows"])
            windows = windows_and_scores[:, :2]
            for func_name in self.process_func_names:
                windows = self.name2func[func_name](windows)
            line["pred_relevant_windows"] = torch.cat(
                [windows, windows_and_scores[:, 2:3]], dim=1
            ).tolist()
            line["pred_relevant_windows"] = [
                e[:2] + [float(f"{e[2]:.4f}")] for e in line["pred_relevant_windows"]
            ]
            processed_lines.append(line)
        return processed_lines

    def clip_min_max_timestamps(self, windows):

        return torch.clamp(windows, min=self.min_ts_val, max=self.max_ts_val)

    def round_to_multiple_clip_lengths(self, windows):

        return torch.round(windows / self.clip_length) * self.clip_length

#### Criterion

In [45]:
@torch.no_grad()
def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k
    output: (#items, #classes)
    target: int,
    """
    maxk = max(topk)
    num_items = output.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target)

    res = []
    for k in topk:
        correct_k = correct[:k].view(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / num_items))
    return res

In [46]:
class SetCriterion(nn.Module):
    """This class computes the loss for DETR.
    The process happens in two steps:
        1) we compute hungarian assignment between ground truth boxes and the outputs of the model
        2) we supervise each pair of matched ground-truth / prediction (supervise class and box)
    """

    def __init__(
        self,
        matcher,
        weight_dict,
        eos_coef,
        losses,
        span_loss_type,
        max_a_l,
        saliency_margin=1,
    ):
        """Create the criterion.
        Parameters:
            matcher: module able to compute a matching between targets and proposals
            weight_dict: dict containing as key the names of the losses and as values their relative weight.
            eos_coef: relative classification weight applied to the no-object category
            losses: list of all the losses to be applied. See get_loss for list of available losses.
            span_loss_type: str, [l1, ce]
            max_v_l: int,
            saliency_margin: float
        """
        super().__init__()
        self.matcher = matcher
        self.weight_dict = weight_dict
        self.losses = losses
        self.span_loss_type = span_loss_type
        self.max_a_l = max_a_l
        self.saliency_margin = saliency_margin

        # foreground and background classification
        self.foreground_label = 0
        self.background_label = 1
        self.eos_coef = eos_coef
        empty_weight = torch.ones(2)
        empty_weight[-1] = (
            self.eos_coef
        )  # lower weight for background (index 1, foreground index 0)
        self.register_buffer("empty_weight", empty_weight)

    def loss_spans(self, outputs, targets, indices):
        """Compute the losses related to the bounding boxes, the L1 regression loss and the GIoU loss
        targets dicts must contain the key "spans" containing a tensor of dim [nb_tgt_spans, 2]
        The target spans are expected in format (center_x, w), normalized by the image size.
        """
        assert "pred_spans" in outputs
        targets = targets["span_labels"]
        idx = self._get_src_permutation_idx(indices)
        src_spans = outputs["pred_spans"][idx]  # (#spans, max_v_l * 2)
        tgt_spans = torch.cat(
            [t["spans"][i] for t, (_, i) in zip(targets, indices)], dim=0
        )  # (#spans, 2)
        if self.span_loss_type == "l1":
            loss_span = F.l1_loss(src_spans, tgt_spans, reduction="none")
            loss_giou = 1 - torch.diag(
                generalized_temporal_iou(
                    span_cxw_to_xx(src_spans), span_cxw_to_xx(tgt_spans)
                )
            )
        else:  # ce
            n_spans = src_spans.shape[0]
            src_spans = src_spans.view(n_spans, 2, self.max_v_l).transpose(1, 2)
            loss_span = F.cross_entropy(src_spans, tgt_spans, reduction="none")
            loss_giou = loss_span.new_zeros([1])

        losses = {}
        losses["loss_span"] = loss_span.mean()
        losses["loss_giou"] = loss_giou.mean()
        return losses

    def loss_labels(self, outputs, targets, indices, log=True):
        """Classification loss (NLL)
        targets dicts must contain the key "labels" containing a tensor of dim [nb_target_boxes]
        """
        # TODO add foreground and background classifier.  use all non-matched as background.
        assert "pred_logits" in outputs
        src_logits = outputs["pred_logits"]  # (batch_size, #queries, #classes=2)
        # idx is a tuple of two 1D tensors (batch_idx, src_idx), of the same length == #objects in batch
        idx = self._get_src_permutation_idx(indices)
        target_classes = torch.full(
            src_logits.shape[:2],
            self.background_label,
            dtype=torch.int64,
            device=src_logits.device,
        )  # (batch_size, #queries)
        target_classes[idx] = self.foreground_label

        loss_ce = F.cross_entropy(
            src_logits.transpose(1, 2),
            target_classes,
            self.empty_weight,
            reduction="none",
        )
        losses = {"loss_label": loss_ce.mean()}

        if log:
            # TODO this should probably be a separate loss, not hacked in this one here
            losses["class_error"] = (
                100 - accuracy(src_logits[idx], self.foreground_label)[0]
            )
        return losses

    def loss_saliency(self, outputs, targets, indices, log=True):
        """higher scores for positive clips"""
        if "saliency_pos_labels" not in targets:
            return {"loss_saliency": 0}

        aud_token_mask = outputs["audio_mask"]

        # Neg pair loss
        saliency_scores_neg = outputs["saliency_scores_neg"].clone()  # (N, L)

        loss_neg_pair = (
            (-torch.log(1.0 - torch.sigmoid(saliency_scores_neg)) * aud_token_mask)
            .sum(dim=1)
            .mean()
        )

        saliency_scores = outputs["saliency_scores"].clone()  # (N, L)
        saliency_contrast_label = targets["saliency_all_labels"]

        saliency_scores = torch.cat([saliency_scores, saliency_scores_neg], dim=1)
        saliency_contrast_label = torch.cat(
            [saliency_contrast_label, torch.zeros_like(saliency_contrast_label)], dim=1
        )

        aud_token_mask = aud_token_mask.repeat([1, 2])
        saliency_scores = (
            aud_token_mask * saliency_scores + (1.0 - aud_token_mask) * -1e3
        )

        tau = 0.5
        loss_rank_contrastive = 0.0

        # for rand_idx in range(1, 13, 3):
        #     # 1, 4, 7, 10 --> 5 stages
        for rand_idx in range(1, 12):
            drop_mask = ~(saliency_contrast_label > 100)  # no drop
            pos_mask = (
                saliency_contrast_label >= rand_idx
            )  # positive when equal or higher than rand_idx

            if torch.sum(pos_mask) == 0:  # no positive sample
                continue
            else:
                batch_drop_mask = (
                    torch.sum(pos_mask, dim=1) > 0
                )  # negative sample indicator

            # drop higher ranks
            cur_saliency_scores = saliency_scores * drop_mask / tau + ~drop_mask * -1e3

            # numerical stability
            logits = (
                cur_saliency_scores
                - torch.max(cur_saliency_scores, dim=1, keepdim=True)[0]
            )

            # softmax
            exp_logits = torch.exp(logits)
            log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-6)

            mean_log_prob_pos = (pos_mask * log_prob * aud_token_mask).sum(1) / (
                pos_mask.sum(1) + 1e-6
            )

            loss = -mean_log_prob_pos * batch_drop_mask

            loss_rank_contrastive = loss_rank_contrastive + loss.mean()

        loss_rank_contrastive = loss_rank_contrastive / 12

        saliency_scores = outputs["saliency_scores"]  # (N, L)
        pos_indices = targets["saliency_pos_labels"]  # (N, #pairs)
        neg_indices = targets["saliency_neg_labels"]  # (N, #pairs)
        num_pairs = pos_indices.shape[1]  # typically 2 or 4
        batch_indices = torch.arange(len(saliency_scores)).to(saliency_scores.device)
        pos_scores = torch.stack(
            [
                saliency_scores[batch_indices, pos_indices[:, col_idx]]
                for col_idx in range(num_pairs)
            ],
            dim=1,
        )
        neg_scores = torch.stack(
            [
                saliency_scores[batch_indices, neg_indices[:, col_idx]]
                for col_idx in range(num_pairs)
            ],
            dim=1,
        )
        loss_saliency = (
            torch.clamp(self.saliency_margin + neg_scores - pos_scores, min=0).sum()
            / (len(pos_scores) * num_pairs)
            * 2
        )  # * 2 to keep the loss the same scale

        loss_saliency = loss_saliency + loss_rank_contrastive + loss_neg_pair
        return {"loss_saliency": loss_saliency}

    def _get_src_permutation_idx(self, indices):
        """Permutes predictions following the given indices.

        Args:
            indices: List of tuples (src, tgt) indices.

        Returns:
            tuple: batch_idx, src_idx
        """
        # permute predictions following indices
        batch_idx = torch.cat(
            [torch.full_like(src, i) for i, (src, _) in enumerate(indices)]
        )
        src_idx = torch.cat([src for (src, _) in indices])
        return batch_idx, src_idx  # two 1D tensors of the same length

    def _get_tgt_permutation_idx(self, indices):
        """Permutes targets following the given indices.

        Args:
            indices: List of tuples (src, tgt) indices.

        Returns:
            tuple: batch_idx, tgt_idx
        """
        # permute targets following indices
        batch_idx = torch.cat(
            [torch.full_like(tgt, i) for i, (_, tgt) in enumerate(indices)]
        )
        tgt_idx = torch.cat([tgt for (_, tgt) in indices])
        return batch_idx, tgt_idx

    def get_loss(self, loss, outputs, targets, indices, **kwargs):
        """Retrieves the loss function for the given loss type.

        Args:
            loss (str): Type of loss.
            outputs: Model outputs.
            targets: Ground truth targets.
            indices: Matched indices.
            **kwargs: Additional arguments.

        Returns:
            dict: Loss values.
        """
        loss_map = {
            "spans": self.loss_spans,
            "labels": self.loss_labels,
            "saliency": self.loss_saliency,
        }
        assert loss in loss_map, f"do you really want to compute {loss} loss?"
        return loss_map[loss](outputs, targets, indices, **kwargs)

    def forward(self, outputs, targets):
        """This performs the loss computation.
        Parameters:
             outputs: dict of tensors, see the output specification of the model for the format
             targets: list of dicts, such that len(targets) == batch_size.
                      The expected keys in each dict depends on the losses applied, see each loss' doc
        """
        outputs_without_aux = {k: v for k, v in outputs.items() if k != "aux_outputs"}

        # Retrieve the matching between the outputs of the last layer and the targets
        # list(tuples), each tuple is (pred_span_indices, tgt_span_indices)

        indices = self.matcher(outputs_without_aux, targets)
        losses_target = self.losses

        # Compute all the requested losses
        losses = {}
        for loss in losses_target:
            losses.update(self.get_loss(loss, outputs, targets, indices))

        # In case of auxiliary losses, we repeat this process with the output of each intermediate layer.
        if "aux_outputs" in outputs:
            for i, aux_outputs in enumerate(outputs["aux_outputs"]):
                indices = self.matcher(aux_outputs, targets)
                losses_target = self.losses

                for loss in losses_target:
                    if "saliency" == loss:  # skip as it is only in the top layer
                        continue
                    kwargs = {}
                    l_dict = self.get_loss(
                        loss, aux_outputs, targets, indices, **kwargs
                    )
                    l_dict = {k + f"_{i}": v for k, v in l_dict.items()}
                    losses.update(l_dict)
        return losses

#### Matcher

In [47]:
from scipy.optimize import linear_sum_assignment


class HungarianMatcher(nn.Module):
    """This class computes an assignment between the targets and the predictions of the network

    For efficiency reasons, the targets don't include the no_object. Because of this, in general,
    there are more predictions than targets. In this case, we do a 1-to-1 matching of the best predictions,
    while the others are un-matched (and thus treated as non-objects).
    """

    def __init__(
        self,
        cost_class: float = 1,
        cost_span: float = 1,
        cost_giou: float = 1,
        span_loss_type: str = "l1",
        max_a_l: int = 75,
    ):
        """Creates the matcher

        Params:
            cost_span: This is the relative weight of the L1 error of the span coordinates in the matching cost
            cost_giou: This is the relative weight of the giou loss of the spans in the matching cost
        """
        super().__init__()
        self.cost_class = cost_class
        self.cost_span = cost_span
        self.cost_giou = cost_giou
        self.span_loss_type = span_loss_type
        self.max_a_l = max_a_l
        self.foreground_label = 0
        assert cost_class != 0 or cost_span != 0 or cost_giou != 0, (
            "all costs cant be 0"
        )

    @torch.no_grad()
    def forward(self, outputs, targets):
        """Performs the matching

        Params:
            outputs: This is a dict that contains at least these entries:
                 "pred_spans": Tensor of dim [batch_size, num_queries, 2] with the predicted span coordinates,
                    in normalized (cx, w) format
                 ""pred_logits": Tensor of dim [batch_size, num_queries, num_classes] with the classification logits

            targets: This is a list of targets (len(targets) = batch_size), where each target is a dict containing:
                 "spans": Tensor of dim [num_target_spans, 2] containing the target span coordinates. The spans are
                    in normalized (cx, w) format

        Returns:
            A list of size batch_size, containing tuples of (index_i, index_j) where:
                - index_i is the indices of the selected predictions (in order)
                - index_j is the indices of the corresponding selected targets (in order)
            For each batch element, it holds:
                len(index_i) = len(index_j) = min(num_queries, num_target_spans)
        """
        bs, num_queries = outputs["pred_spans"].shape[:2]
        targets = targets["span_labels"]

        # Also concat the target labels and spans
        out_prob = (
            outputs["pred_logits"].flatten(0, 1).softmax(-1)
        )  # [batch_size * num_queries, num_classes]
        tgt_spans = torch.cat(
            [v["spans"] for v in targets]
        )  # [num_target_spans in batch, 2]
        tgt_ids = torch.full(
            [len(tgt_spans)], self.foreground_label
        )  # [total #spans in the batch]

        # Compute the classification cost. Contrary to the loss, we don't use the NLL,
        # but approximate it in 1 - prob[target class].
        # The 1 is a constant that doesn't change the matching, it can be omitted.
        cost_class = -out_prob[
            :, tgt_ids
        ]  # [batch_size * num_queries, total #spans in the batch]

        if self.span_loss_type == "l1":
            # We flatten to compute the cost matrices in a batch
            out_spans = outputs["pred_spans"].flatten(
                0, 1
            )  # [batch_size * num_queries, 2]

            # Compute the L1 cost between spans
            cost_span = torch.cdist(
                out_spans, tgt_spans, p=1
            )  # [batch_size * num_queries, total #spans in the batch]

            # Compute the giou cost between spans
            # [batch_size * num_queries, total #spans in the batch]
            cost_giou = -generalized_temporal_iou(
                span_cxw_to_xx(out_spans), span_cxw_to_xx(tgt_spans)
            )
        else:
            pred_spans = outputs["pred_spans"]  # (bsz, #queries, max_v_l * 2)
            pred_spans = pred_spans.view(bs * num_queries, 2, self.max_v_l).softmax(
                -1
            )  # (bsz * #queries, 2, max_v_l)
            cost_span = (
                -pred_spans[:, 0][:, tgt_spans[:, 0]]
                - pred_spans[:, 1][:, tgt_spans[:, 1]]
            )  # (bsz * #queries, #spans)
            # pred_spans = pred_spans.repeat(1, n_spans, 1, 1).flatten(0, 1)  # (bsz * #queries * #spans, max_v_l, 2)
            # tgt_spans = tgt_spans.view(1, n_spans, 2).repeat(bs * num_queries, 1, 1).flatten(0, 1)  # (bsz * #queries * #spans, 2)
            # cost_span = pred_spans[tgt_spans]
            # cost_span = cost_span.view(bs * num_queries, n_spans)

            # giou
            cost_giou = 0

        # Final cost matrix
        C = (
            self.cost_span * cost_span
            + self.cost_giou * cost_giou
            + self.cost_class * cost_class
        )
        C = C.view(bs, num_queries, -1).cpu()

        sizes = [len(v["spans"]) for v in targets]
        indices = [
            linear_sum_assignment(c[i]) for i, c in enumerate(C.split(sizes, -1))
        ]
        return [
            (
                torch.as_tensor(i, dtype=torch.int64),
                torch.as_tensor(j, dtype=torch.int64),
            )
            for i, j in indices
        ]


class HungarianEventMatcher(nn.Module):
    """This class computes an assignment between the targets and the predictions of the network

    For efficiency reasons, the targets don't include the no_object. Because of this, in general,
    there are more predictions than targets. In this case, we do a 1-to-1 matching of the best predictions,
    while the others are un-matched (and thus treated as non-objects).
    """

    def __init__(
        self,
        cost_span: float = 1,
        cost_giou: float = 1,
        span_loss_type: str = "l1",
        max_v_l: int = 75,
    ):
        """Creates the matcher

        Params:
            cost_span: This is the relative weight of the L1 error of the span coordinates in the matching cost
            cost_giou: This is the relative weight of the giou loss of the spans in the matching cost
        """
        super().__init__()
        self.cost_span = cost_span
        self.cost_giou = cost_giou
        self.span_loss_type = span_loss_type
        self.max_v_l = max_v_l
        self.foreground_label = 0
        assert cost_span != 0 or cost_giou != 0, "all costs cant be 0"

    @torch.no_grad()
    def forward(self, outputs, targets):
        """Performs the matching

        Params:
            outputs: This is a dict that contains at least these entries:
                 "pred_spans": Tensor of dim [batch_size, num_queries, 2] with the predicted span coordinates,
                    in normalized (cx, w) format
                 ""pred_logits": Tensor of dim [batch_size, num_queries, num_classes] with the classification logits

            targets: This is a list of targets (len(targets) = batch_size), where each target is a dict containing:
                 "spans": Tensor of dim [num_target_spans, 2] containing the target span coordinates. The spans are
                    in normalized (cx, w) format

        Returns:
            A list of size batch_size, containing tuples of (index_i, index_j) where:
                - index_i is the indices of the selected predictions (in order)
                - index_j is the indices of the corresponding selected targets (in order)
            For each batch element, it holds:
                len(index_i) = len(index_j) = min(num_queries, num_target_spans)
        """
        bs, num_queries = outputs.shape[:2]

        # Also concat the target labels and spans
        tgt_spans = torch.cat([v for v in targets])  # [num_target_spans in batch, 2]

        # We flatten to compute the cost matrices in a batch
        out_spans = outputs.flatten(0, 1)  # [batch_size * num_queries, 2]

        # Compute the L1 cost between spans
        cost_span = torch.cdist(
            out_spans, tgt_spans, p=1
        )  # [batch_size * num_queries, total #spans in the batch]

        # Compute the giou cost between spans
        # [batch_size * num_queries, total #spans in the batch]
        cost_giou = -generalized_temporal_iou_(
            span_cxw_to_xx(out_spans), span_cxw_to_xx(tgt_spans)
        )

        # Final cost matrix
        C = self.cost_span * cost_span + self.cost_giou * cost_giou
        C = C.view(bs, num_queries, -1).cpu()

        sizes = [len(v) for v in targets]
        indices = [
            linear_sum_assignment(c[i]) for i, c in enumerate(C.split(sizes, -1))
        ]
        return [
            (
                torch.as_tensor(i, dtype=torch.int64),
                torch.as_tensor(j, dtype=torch.int64),
            )
            for i, j in indices
        ]


def build_matcher(args):
    return HungarianMatcher(
        cost_span=args.set_cost_span,
        cost_giou=args.set_cost_giou,
        cost_class=args.set_cost_class,
        span_loss_type=args.span_loss_type,
        max_a_l=args.max_a_l,
    )


def build_event_matcher(args):
    return HungarianEventMatcher(
        cost_span=args.set_cost_span,
        cost_giou=args.set_cost_giou,
        span_loss_type=args.span_loss_type,
        max_a_l=args.max_a_l,
    )

In [138]:
def build_model(args):
    # the `num_classes` naming here is somewhat misleading.
    # it indeed corresponds to `max_obj_id + 1`, where max_obj_id
    # is the maximum id for a class in your dataset. For example,
    # COCO has a max_obj_id of 90, so we pass `num_classes` to be 91.
    # As another example, for a dataset that has a single class with id 1,
    # you should pass `num_classes` to be 2 (max_obj_id + 1).
    # For more details on this, check the following discussion
    # https://github.com/facebookresearch/qd_detr/issues/108#issuecomment-650269223
    device = torch.device(args.device)
    transformer = build_transformer(args)
    position_embedding, txt_position_embedding = build_position_encoding(args)

    model = QDDETR(
        transformer=transformer,
        position_embed=position_embedding,
        txt_position_embed=txt_position_embedding,
        max_a_l=args.max_a_l,
        txt_dim=args.t_feat_dim,
        aud_dim=args.a_feat_dim,
        aux_loss=args.aux_loss,
        num_queries=args.num_queries,
        input_dropout=args.input_dropout,
        span_loss_type=args.span_loss_type,
        n_input_proj=args.n_input_proj,
    )

    matcher = build_matcher(args)
    weight_dict = {
        "loss_span": args.span_loss_coef,
        "loss_giou": args.giou_loss_coef,
        "loss_label": args.label_loss_coef,
        "loss_saliency": args.lw_saliency,
    }

    if args.aux_loss:
        aux_weight_dict = {}
        for i in range(args.dec_layers - 1):
            aux_weight_dict.update(
                {k + f"_{i}": v for k, v in weight_dict.items() if k != "loss_saliency"}
            )
        weight_dict.update(aux_weight_dict)

    losses = ["spans", "labels", "saliency"]
    criterion = SetCriterion(
        matcher=matcher,
        weight_dict=weight_dict,
        losses=losses,
        eos_coef=args.eos_coef,
        span_loss_type=args.span_loss_type,
        max_a_l=args.max_a_l,
        saliency_margin=args.saliency_margin,
    )
    criterion.to(device)
    return model, criterion

### Training


In [49]:
class ModelEMA(torch.nn.Module):
    def __init__(self, model, decay=0.999, device=None):
        super().__init__()
        # make a copy of the model for accumulating moving average of weights
        self.module = copy.deepcopy(model)
        self.module.eval()
        self.decay = decay
        self.device = device  # perform ema on different device from model if set
        if self.device is not None:
            self.module.to(device=device)

    def _update(self, model, update_fn):
        with torch.no_grad():
            for ema_v, model_v in zip(
                self.module.state_dict().values(), model.state_dict().values()
            ):
                if self.device is not None:
                    model_v = model_v.to(device=self.device)
                ema_v.copy_(update_fn(ema_v, model_v))

    def update(self, model):
        self._update(
            model, update_fn=lambda e, m: self.decay * e + (1.0 - self.decay) * m
        )

    def set(self, model):
        self._update(model, update_fn=lambda e, m: m)

In [107]:
def setup_model(opt):
    """setup model/optimizer/scheduler and load checkpoints when needed"""
    logger.info("setup model/optimizer/scheduler")
    # model, criterion = build_model_qd_detr(opt)
    model, criterion = build_model(opt)

    if opt.device == "cuda" or opt.device == 'mps':
        logger.info("CUDA/MPS enabled.")
        model.to(opt.device)
        criterion.to(opt.device)

    param_dicts = [
        {"params": [p for n, p in model.named_parameters() if p.requires_grad]}
    ]
    optimizer = torch.optim.AdamW(param_dicts, lr=opt.lr, weight_decay=opt.wd)
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, opt.lr_drop)

    return model, criterion, optimizer, lr_scheduler

##### Counte/Compute (1)

In [108]:
from sklearn.metrics import precision_recall_curve


def compute_temporal_iou_batch_paired(pred_windows, gt_windows):
    """compute intersection-over-union along temporal axis for each pair of windows in pred_windows and gt_windows.
    Args:
        pred_windows: np.ndarray, (N, 2), [st (float), ed (float)] * N
        gt_windows: np.ndarray, (N, 2), [st (float), ed (float)] * N
    Returns:
        iou (float): np.ndarray, (N, )

    References:
        for np.divide with zeros, see https://stackoverflow.com/a/37977222
    """
    intersection = np.maximum(
        0,
        np.minimum(pred_windows[:, 1], gt_windows[:, 1])
        - np.maximum(pred_windows[:, 0], gt_windows[:, 0]),
    )
    union = np.maximum(pred_windows[:, 1], gt_windows[:, 1]) - np.minimum(
        pred_windows[:, 0], gt_windows[:, 0]
    )  # not the correct union though
    return np.divide(
        intersection, union, out=np.zeros_like(intersection), where=union != 0
    )


def compute_temporal_iou_batch_cross(spans1, spans2):
    """
    Args:
        spans1: (N, 2) np.ndarray, each row defines a span [st, ed]
        spans2: (M, 2) np.ndarray, ...

    Returns:
        iou: (N, M) np.ndarray
        union: (N, M) np.ndarray
    >>> spans1 = np.array([[0, 0.2, 0.9], [0.5, 1.0, 0.2]])
    >>> spans2 = np.array([[0, 0.3], [0., 1.0]])
    >>> compute_temporal_iou_batch_cross(spans1, spans2)
    (tensor([[0.6667, 0.2000],
         [0.0000, 0.5000]]),
     tensor([[0.3000, 1.0000],
             [0.8000, 1.0000]]))
    """
    areas1 = spans1[:, 1] - spans1[:, 0]  # (N, )
    areas2 = spans2[:, 1] - spans2[:, 0]  # (M, )

    left = np.maximum(spans1[:, None, 0], spans2[None, :, 0])  # (N, M)
    right = np.minimum(spans1[:, None, 1], spans2[None, :, 1])  # (N, M)

    inter = np.clip(right - left, 0, None)  # (N, M)
    union = areas1[:, None] + areas2[None, :] - inter  # (N, M)

    iou = inter / union
    return iou, union


def interpolated_precision_recall(precision, recall):
    """Interpolated AP - VOCdevkit from VOC 2011.

    Args:
        precision (np.ndarray): The precision of different thresholds.
        recall (np.ndarray): The recall of different thresholds.

    Returns：
        float: Average precision score.
    """
    mprecision = np.hstack([[0], precision, [0]])
    mrecall = np.hstack([[0], recall, [1]])
    for i in range(len(mprecision) - 1)[::-1]:
        mprecision[i] = max(mprecision[i], mprecision[i + 1])
    idx = np.where(mrecall[1::] != mrecall[0:-1])[0] + 1
    ap = np.sum((mrecall[idx] - mrecall[idx - 1]) * mprecision[idx])
    return ap


def compute_average_precision_detection(
    ground_truth, prediction, tiou_thresholds=np.linspace(0.5, 0.95, 10)
):
    """Compute average precision (detection task) between ground truth and
    predictions data frames. If multiple predictions occurs for the same
    predicted segment, only the one with highest score is matches as true
    positive. This code is greatly inspired by Pascal VOC devkit.

    Args:
        ground_truth (list[dict]): List containing the ground truth instances
            (dictionaries). Required keys are 'video-id', 't-start' and
            't-end'.
        prediction (list[dict]): List containing the prediction instances
            (dictionaries). Required keys are: 'video-id', 't-start', 't-end'
            and 'score'.
        tiou_thresholds (np.ndarray): A 1darray indicates the temporal
            intersection over union threshold, which is optional.
            Default: ``np.linspace(0.5, 0.95, 10)``.

    Returns:
        Float: ap, Average precision score.
    """
    num_thresholds = len(tiou_thresholds)
    num_gts = len(ground_truth)
    num_preds = len(prediction)
    ap = np.zeros(num_thresholds)
    if len(prediction) == 0:
        return ap

    num_positive = float(num_gts)
    lock_gt = np.ones((num_thresholds, num_gts)) * -1
    # Sort predictions by decreasing score order.
    prediction.sort(key=lambda x: -x["score"])
    # Initialize true positive and false positive vectors.
    tp = np.zeros((num_thresholds, num_preds))
    fp = np.zeros((num_thresholds, num_preds))

    # Adaptation to query faster
    ground_truth_by_videoid = {}
    for i, item in enumerate(ground_truth):
        item["index"] = i
        ground_truth_by_videoid.setdefault(item["video-id"], []).append(item)

    # Assigning true positive to truly grount truth instances.
    for idx, pred in enumerate(prediction):
        if pred["video-id"] in ground_truth_by_videoid:
            gts = ground_truth_by_videoid[pred["video-id"]]
        else:
            fp[:, idx] = 1
            continue

        _pred = np.array(
            [
                [pred["t-start"], pred["t-end"]],
            ]
        )
        _gt = np.array([[gt["t-start"], gt["t-end"]] for gt in gts])
        tiou_arr = compute_temporal_iou_batch_cross(_pred, _gt)[0]

        tiou_arr = tiou_arr.reshape(-1)
        # We would like to retrieve the predictions with highest tiou score.
        tiou_sorted_idx = tiou_arr.argsort()[::-1]
        for t_idx, tiou_threshold in enumerate(tiou_thresholds):
            for j_idx in tiou_sorted_idx:
                if tiou_arr[j_idx] < tiou_threshold:
                    fp[t_idx, idx] = 1
                    break
                if lock_gt[t_idx, gts[j_idx]["index"]] >= 0:
                    continue
                # Assign as true positive after the filters above.
                tp[t_idx, idx] = 1
                lock_gt[t_idx, gts[j_idx]["index"]] = idx
                break

            if fp[t_idx, idx] == 0 and tp[t_idx, idx] == 0:
                fp[t_idx, idx] = 1

    tp_cumsum = np.cumsum(tp, axis=1).astype(float)
    fp_cumsum = np.cumsum(fp, axis=1).astype(float)
    recall_cumsum = tp_cumsum / num_positive

    precision_cumsum = tp_cumsum / (tp_cumsum + fp_cumsum)

    for t_idx in range(len(tiou_thresholds)):
        ap[t_idx] = interpolated_precision_recall(
            precision_cumsum[t_idx, :], recall_cumsum[t_idx, :]
        )
    return ap


def get_ap(y_true, y_predict, interpolate=True, point_11=False):
    """
    Average precision in different formats: (non-) interpolated and/or 11-point approximated
    point_11=True and interpolate=True corresponds to the 11-point interpolated AP used in
    the PASCAL VOC challenge up to the 2008 edition and has been verfied against the vlfeat implementation
    The exact average precision (interpolate=False, point_11=False) corresponds to the one of vl_feat

    :param y_true: list/ numpy vector of true labels in {0,1} for each element
    :param y_predict: predicted score for each element
    :param interpolate: Use interpolation?
    :param point_11: Use 11-point approximation to average precision?
    :return: average precision

    ref: https://github.com/gyglim/video2gif_dataset/blob/master/v2g_evaluation/__init__.py

    """
    # Check inputs
    assert len(y_true) == len(y_predict), (
        "Prediction and ground truth need to be of the same length"
    )
    if len(set(y_true)) == 1:
        if y_true[0] == 0:
            return 0  # True labels are all zeros
            # raise ValueError('True labels cannot all be zero')
        else:
            return 1
    else:
        assert sorted(set(y_true)) == [0, 1], (
            "Ground truth can only contain elements {0,1}"
        )

    # Compute precision and recall
    precision, recall, _ = precision_recall_curve(y_true, y_predict)
    recall = recall.astype(np.float32)

    if interpolate:  # Compute the interpolated precision
        for i in range(1, len(precision)):
            precision[i] = max(precision[i - 1], precision[i])

    if point_11:  # Compute the 11-point approximated AP
        precision_11 = [
            precision[np.where(recall >= t)[0][-1]] for t in np.arange(0, 1.01, 0.1)
        ]
        return np.mean(precision_11)
    else:  # Compute the AP using precision at every additionally recalled sample
        indices = np.where(np.diff(recall))
        return np.mean(precision[indices])

##### Count/Compute (2)

In [52]:
def compute_average_precision_detection_wrapper(
    input_triple, tiou_thresholds=np.linspace(0.5, 0.95, 10)
):
    qid, ground_truth, prediction = input_triple
    scores = compute_average_precision_detection(
        ground_truth, prediction, tiou_thresholds=tiou_thresholds
    )
    return qid, scores


def compute_mr_ap(
    submission,
    ground_truth,
    iou_thds=np.linspace(0.5, 0.95, 10),
    max_gt_windows=None,
    max_pred_windows=10,
    num_workers=8,
    chunksize=50,
):
    iou_thds = [float(f"{e:.2f}") for e in iou_thds]
    pred_qid2data = defaultdict(list)
    for d in submission:
        pred_windows = (
            d["pred_relevant_windows"][:max_pred_windows]
            if max_pred_windows is not None
            else d["pred_relevant_windows"]
        )
        qid = d["qid"]
        for w in pred_windows:
            pred_qid2data[qid].append(
                {
                    "video-id": d["qid"],  # in order to use the API
                    "t-start": w[0],
                    "t-end": w[1],
                    "score": w[2],
                }
            )

    gt_qid2data = defaultdict(list)
    for d in ground_truth:
        gt_windows = (
            d["relevant_windows"][:max_gt_windows]
            if max_gt_windows is not None
            else d["relevant_windows"]
        )
        qid = d["qid"]
        for w in gt_windows:
            gt_qid2data[qid].append(
                {"video-id": d["qid"], "t-start": w[0], "t-end": w[1]}
            )
    qid2ap_list = {}
    # start_time = time.time()
    data_triples = [
        [qid, gt_qid2data[qid], pred_qid2data[qid]] for qid in pred_qid2data
    ]
    from functools import partial

    compute_ap_from_triple = partial(
        compute_average_precision_detection_wrapper, tiou_thresholds=iou_thds
    )

    if num_workers > 1:
        with mp.Pool(num_workers) as pool:
            for qid, scores in pool.imap_unordered(
                compute_ap_from_triple, data_triples, chunksize=chunksize
            ):
                qid2ap_list[qid] = scores
    else:
        for data_triple in data_triples:
            qid, scores = compute_ap_from_triple(data_triple)
            qid2ap_list[qid] = scores

    # print(f"compute_average_precision_detection {time.time() - start_time:.2f} seconds.")
    ap_array = np.array(list(qid2ap_list.values()))  # (#queries, #thd)
    ap_thds = ap_array.mean(0)  # mAP at different IoU thresholds.
    iou_thd2ap = dict(zip([str(e) for e in iou_thds], ap_thds))
    iou_thd2ap["average"] = np.mean(ap_thds)
    # formatting
    iou_thd2ap = {k: float(f"{100 * v:.2f}") for k, v in iou_thd2ap.items()}
    return iou_thd2ap


def compute_mr_r1(submission, ground_truth, iou_thds=np.linspace(0.5, 0.95, 10)):
    """If a predicted segment has IoU >= iou_thd with one of the 1st GT segment, we define it positive"""
    iou_thds = [float(f"{e:.2f}") for e in iou_thds]
    pred_qid2window = {
        d["qid"]: d["pred_relevant_windows"][0][:2] for d in submission
    }  # :2 rm scores
    # gt_qid2window = {d["qid"]: d["relevant_windows"][0] for d in ground_truth}
    gt_qid2window = {}
    for d in ground_truth:
        cur_gt_windows = d["relevant_windows"]
        cur_qid = d["qid"]
        cur_max_iou_idx = 0
        if len(cur_gt_windows) > 0:  # select the GT window that has the highest IoU
            cur_ious = compute_temporal_iou_batch_cross(
                np.array([pred_qid2window[cur_qid]]), np.array(d["relevant_windows"])
            )[0]
            cur_max_iou_idx = np.argmax(cur_ious)
        gt_qid2window[cur_qid] = cur_gt_windows[cur_max_iou_idx]

    qids = list(pred_qid2window.keys())
    pred_windows = np.array([pred_qid2window[k] for k in qids]).astype(float)
    gt_windows = np.array([gt_qid2window[k] for k in qids]).astype(float)
    pred_gt_iou = compute_temporal_iou_batch_paired(pred_windows, gt_windows)
    iou_thd2recall_at_one = {}
    for thd in iou_thds:
        iou_thd2recall_at_one[str(thd)] = float(
            f"{np.mean(pred_gt_iou >= thd) * 100:.2f}"
        )
    return iou_thd2recall_at_one


def get_window_len(window):
    return window[1] - window[0]


def get_data_by_range(submission, ground_truth, len_range):
    """keep queries with ground truth window length in the specified length range.
    Args:
        submission:
        ground_truth:
        len_range: [min_l (int), max_l (int)]. the range is (min_l, max_l], i.e., min_l < l <= max_l
    """
    min_l, max_l = len_range
    if min_l == 0 and max_l == 150:  # min and max l in dataset
        return submission, ground_truth

    # only keep ground truth with windows in the specified length range
    # if multiple GT windows exists, we only keep the ones in the range
    ground_truth_in_range = []
    gt_qids_in_range = set()
    for d in ground_truth:
        rel_windows_in_range = [
            w for w in d["relevant_windows"] if min_l < get_window_len(w) <= max_l
        ]
        if len(rel_windows_in_range) > 0:
            d = copy.deepcopy(d)
            d["relevant_windows"] = rel_windows_in_range
            ground_truth_in_range.append(d)
            gt_qids_in_range.add(d["qid"])

    # keep only submissions for ground_truth_in_range
    submission_in_range = []
    for d in submission:
        if d["qid"] in gt_qids_in_range:
            submission_in_range.append(copy.deepcopy(d))

    return submission_in_range, ground_truth_in_range


def eval_moment_retrieval(submission, ground_truth, verbose=True):
    # length_ranges = [[0, 10], [10, 30], [30, 150], [0, 150], ]  #
    # range_names = ["short", "middle", "long", "full"]
    length_ranges = [[0, 1500]]  # TODO: cover all examples?
    range_names = ["full"]

    ret_metrics = {}
    for l_range, name in zip(length_ranges, range_names):
        if verbose:
            start_time = time.time()
        _submission, _ground_truth = get_data_by_range(
            submission, ground_truth, l_range
        )
        print(
            f"{name}: {l_range}, {len(_ground_truth)}/{len(ground_truth)}="
            f"{100 * len(_ground_truth) / len(ground_truth):.2f} examples."
        )
        iou_thd2average_precision = compute_mr_ap(
            _submission, _ground_truth, num_workers=8, chunksize=50
        )
        iou_thd2recall_at_one = compute_mr_r1(_submission, _ground_truth)
        ret_metrics[name] = {
            "MR-mAP": iou_thd2average_precision,
            "MR-R1": iou_thd2recall_at_one,
        }
        if verbose:
            print(
                f"[eval_moment_retrieval] [{name}] {time.time() - start_time:.2f} seconds"
            )
    return ret_metrics


def compute_hl_hit1(qid2preds, qid2gt_scores_binary):
    qid2max_scored_clip_idx = {
        k: np.argmax(v["pred_saliency_scores"]) for k, v in qid2preds.items()
    }
    hit_scores = np.zeros((len(qid2preds), 3))
    qids = list(qid2preds.keys())
    for idx, qid in enumerate(qids):
        pred_clip_idx = qid2max_scored_clip_idx[qid]
        gt_scores_binary = qid2gt_scores_binary[qid]  # (#clips, 3)
        if pred_clip_idx < len(gt_scores_binary):
            hit_scores[idx] = gt_scores_binary[pred_clip_idx]
    # aggregate scores from 3 separate annotations (3 workers) by taking the max.
    # then average scores from all queries.
    hit_at_one = float(f"{100 * np.mean(np.max(hit_scores, 1)):.2f}")
    return hit_at_one


def compute_hl_ap(qid2preds, qid2gt_scores_binary, num_workers=8, chunksize=50):
    qid2pred_scores = {k: v["pred_saliency_scores"] for k, v in qid2preds.items()}
    ap_scores = np.zeros((len(qid2preds), 3))  # (#preds, 3)
    qids = list(qid2preds.keys())
    input_tuples = []
    for idx, qid in enumerate(qids):
        for w_idx in range(3):  # annotation score idx
            y_true = qid2gt_scores_binary[qid][:, w_idx]
            y_predict = np.array(qid2pred_scores[qid])
            input_tuples.append((idx, w_idx, y_true, y_predict))

    if num_workers > 1:
        with mp.Pool(num_workers) as pool:
            for idx, w_idx, score in pool.imap_unordered(
                compute_ap_from_tuple, input_tuples, chunksize=chunksize
            ):
                ap_scores[idx, w_idx] = score
    else:
        for input_tuple in input_tuples:
            idx, w_idx, score = compute_ap_from_tuple(input_tuple)
            ap_scores[idx, w_idx] = score

    # it's the same if we first average across different annotations, then average across queries
    # since all queries have the same #annotations.
    mean_ap = float(f"{100 * np.mean(ap_scores):.2f}")
    return mean_ap


def compute_ap_from_tuple(input_tuple):
    idx, w_idx, y_true, y_predict = input_tuple
    if len(y_true) < len(y_predict):
        # print(f"len(y_true) < len(y_predict) {len(y_true), len(y_predict)}")
        y_predict = y_predict[: len(y_true)]
    elif len(y_true) > len(y_predict):
        # print(f"len(y_true) > len(y_predict) {len(y_true), len(y_predict)}")
        _y_predict = np.zeros(len(y_true))
        _y_predict[: len(y_predict)] = y_predict
        y_predict = _y_predict

    score = get_ap(y_true, y_predict)
    return idx, w_idx, score


def mk_gt_scores(gt_data, clip_length=2):
    """gt_data, dict,"""
    num_clips = int(gt_data["duration"] / clip_length)
    saliency_scores_full_video = np.zeros((num_clips, 3))
    relevant_clip_ids = np.array(gt_data["relevant_clip_ids"])  # (#relevant_clip_ids, )
    saliency_scores_relevant_clips = np.array(
        gt_data["saliency_scores"]
    )  # (#relevant_clip_ids, 3)
    saliency_scores_full_video[relevant_clip_ids] = saliency_scores_relevant_clips
    return saliency_scores_full_video  # (#clips_in_video, 3)  the scores are in range [0, 4]


def eval_highlight(submission, ground_truth, verbose=True):
    """
    Args:
        submission:
        ground_truth:
        verbose:
    """
    qid2preds = {d["qid"]: d for d in submission}
    qid2gt_scores_full_range = {
        d["qid"]: mk_gt_scores(d) for d in ground_truth
    }  # scores in range [0, 4]
    # gt_saliency_score_min: int, in [0, 1, 2, 3, 4]. The minimum score for a positive clip.
    gt_saliency_score_min_list = [2, 3, 4]
    saliency_score_names = ["Fair", "Good", "VeryGood"]
    highlight_det_metrics = {}
    for gt_saliency_score_min, score_name in zip(
        gt_saliency_score_min_list, saliency_score_names
    ):
        start_time = time.time()
        qid2gt_scores_binary = {
            k: (v >= gt_saliency_score_min).astype(float)
            for k, v in qid2gt_scores_full_range.items()
        }  # scores in [0, 1]
        hit_at_one = compute_hl_hit1(qid2preds, qid2gt_scores_binary)
        mean_ap = compute_hl_ap(qid2preds, qid2gt_scores_binary)
        highlight_det_metrics[f"HL-min-{score_name}"] = {
            "HL-mAP": mean_ap,
            "HL-Hit1": hit_at_one,
        }
        if verbose:
            print(
                f"Calculating highlight scores with min score {gt_saliency_score_min} ({score_name})"
            )
            print(f"Time cost {time.time() - start_time:.2f} seconds")
    return highlight_det_metrics


def eval_submission(submission, ground_truth, verbose=True, match_number=True):
    """
    Args:
        submission: list(dict), each dict is {
            qid: str,
            query: str,
            vid: str,
            pred_relevant_windows: list([st, ed]),
            pred_saliency_scores: list(float), len == #clips in video.
                i.e., each clip in the video will have a saliency score.
        }
        ground_truth: list(dict), each dict is     {
          "qid": 7803,
          "query": "Man in gray top walks from outside to inside.",
          "duration": 150,
          "vid": "RoripwjYFp8_360.0_510.0",
          "relevant_clip_ids": [13, 14, 15, 16, 17]
          "saliency_scores": [[4, 4, 2], [3, 4, 2], [2, 2, 3], [2, 2, 2], [0, 1, 3]]
               each sublist corresponds to one clip in relevant_clip_ids.
               The 3 elements in the sublist are scores from 3 different workers. The
               scores are in [0, 1, 2, 3, 4], meaning [Very Bad, ..., Good, Very Good]
        }
        verbose:
        match_number:

    Returns:

    """
    pred_qids = set([e["qid"] for e in submission])
    gt_qids = set([e["qid"] for e in ground_truth])
    if match_number:
        assert pred_qids == gt_qids, (
            f"qids in ground_truth and submission must match. "
            f"use `match_number=False` if you wish to disable this check"
        )
    else:  # only leave the items that exists in both submission and ground_truth
        shared_qids = pred_qids.intersection(gt_qids)
        submission = [e for e in submission if e["qid"] in shared_qids]
        ground_truth = [e for e in ground_truth if e["qid"] in shared_qids]

    eval_metrics = {}
    eval_metrics_brief = OrderedDict()
    if "pred_relevant_windows" in submission[0]:
        moment_ret_scores = eval_moment_retrieval(
            submission, ground_truth, verbose=verbose
        )
        eval_metrics.update(moment_ret_scores)
        moment_ret_scores_brief = {
            "MR-full-mAP": moment_ret_scores["full"]["MR-mAP"]["average"],
            "MR-full-mAP@0.5": moment_ret_scores["full"]["MR-mAP"]["0.5"],
            "MR-full-mAP@0.75": moment_ret_scores["full"]["MR-mAP"]["0.75"],
            "MR-full-R1@0.5": moment_ret_scores["full"]["MR-R1"]["0.5"],
            "MR-full-R1@0.7": moment_ret_scores["full"]["MR-R1"]["0.7"],
        }
        eval_metrics_brief.update(
            sorted(
                [(k, v) for k, v in moment_ret_scores_brief.items()], key=lambda x: x[0]
            )
        )

    if "pred_saliency_scores" in submission[0]:
        highlight_det_scores = eval_highlight(submission, ground_truth, verbose=verbose)
        eval_metrics.update(highlight_det_scores)
        highlight_det_scores_brief = dict(
            [
                (f"{k}-{sub_k.split('-')[1]}", v[sub_k])
                for k, v in highlight_det_scores.items()
                for sub_k in v
            ]
        )
        eval_metrics_brief.update(highlight_det_scores_brief)

    # sort by keys
    final_eval_metrics = OrderedDict()
    final_eval_metrics["brief"] = eval_metrics_brief
    final_eval_metrics.update(
        sorted([(k, v) for k, v in eval_metrics.items()], key=lambda x: x[0])
    )
    return final_eval_metrics

In [53]:
def eval_epoch_post_processing(submission, opt, gt_data, save_submission_filename):
    """Evaluates epoch with post processing.

    Args:
        submission: Submission.
        opt: Options.
        gt_data: Ground truth data.
        save_submission_filename: Filename.

    Returns:
        tuple: metrics, latest_file_paths
    """
    logger.info("Saving/Evaluating before nms results")
    submission_path = os.path.join(opt.results_dir, save_submission_filename)
    save_jsonl(submission, submission_path)

    if opt.eval_split_name in ["val", "test"]:
        metrics = eval_submission(submission, gt_data)
        save_metrics_path = submission_path.replace(".jsonl", "_metrics.json")
        save_json(metrics, save_metrics_path, save_pretty=True, sort_keys=False)
        latest_file_paths = [submission_path, save_metrics_path]
    else:
        metrics = None
        latest_file_paths = [
            submission_path,
        ]

    return metrics, latest_file_paths


@torch.no_grad()
def compute_mr_results(model, eval_loader, opt, criterion=None):
    """Computes MR results.

    Args:
        model: Model.
        eval_loader: Eval loader.
        opt: Options.
        criterion: Criterion.

    Returns:
        tuple: mr_res, loss_meters
    """
    batch_input_fn = (
        cg_detr_prepare_batch_inputs
        if opt.model_name == "cg_detr"
        else prepare_batch_inputs
    )
    loss_meters = defaultdict(AverageMeter)

    mr_res = []
    for batch in tqdm(eval_loader, desc="compute st ed scores"):
        query_meta = batch[0]
        model_inputs, targets = batch_input_fn(batch[1], opt.device)
        outputs = model(**model_inputs)

        # compose predictions
        pred_spans = outputs["pred_spans"].cpu()  # (bsz, #queries, 2)
        prob = F.softmax(
            outputs["pred_logits"], -1
        )  # (batch_size, #queries, #classes=2)
        scores = prob[
            ..., 0
        ].cpu()  # * (batch_size, #queries)  foreground label is 0, we directly take it

        for idx, (meta, spans, score) in enumerate(zip(query_meta, pred_spans, scores)):
            spans = span_cxw_to_xx(spans) * meta["duration"]
            cur_ranked_preds = torch.cat([spans, score[:, None]], dim=1).tolist()
            cur_ranked_preds = sorted(
                cur_ranked_preds, key=lambda x: x[2], reverse=True
            )
            cur_ranked_preds = [
                [float(f"{e:.4f}") for e in row] for row in cur_ranked_preds
            ]

            cur_query_pred = dict(
                qid=meta["qid"],
                query=meta["query"],
                vid=meta["vid"],
                pred_relevant_windows=cur_ranked_preds,
            )

            mr_res.append(cur_query_pred)

        if criterion:
            loss_dict = criterion(outputs, targets)
            weight_dict = criterion.weight_dict
            losses = sum(
                loss_dict[k] * weight_dict[k]
                for k in loss_dict.keys()
                if k in weight_dict
            )
            loss_dict["loss_overall"] = float(losses)
            for k, v in loss_dict.items():
                loss_meters[k].update(
                    float(v) * weight_dict[k] if k in weight_dict else float(v)
                )

    post_processor = PostProcessorDETR(
        clip_length=opt.clip_length,
        min_ts_val=0,
        max_ts_val=300,
        min_w_l=1,
        max_w_l=300,
        move_window_method="left",
        process_func_names=("clip_ts", "round_multiple"),
    )

    mr_res = post_processor(mr_res)
    return mr_res, loss_meters


def get_eval_res(model, eval_loader, opt, criterion):
    """compute and save query and video proposal embeddings"""
    eval_res, eval_loss_meters = compute_mr_results(model, eval_loader, opt, criterion)
    return eval_res, eval_loss_meters


def eval_epoch(model, eval_dataset, opt, save_submission_filename, criterion):
    """Evaluates epoch.

    Args:
        model: Model.
        eval_dataset: Eval dataset.
        opt: Options.
        save_submission_filename: Filename.
        criterion: Criterion.

    Returns:
        tuple: metrics, eval_loss_meters, latest_file_paths
    """
    logger.info("Generate submissions")
    model.eval()
    criterion.eval()

    eval_loader = DataLoader(
        eval_dataset,
        collate_fn=start_end_collate,
        batch_size=opt.eval_bsz,
        num_workers=opt.num_workers,
        shuffle=False,
    )

    submission, eval_loss_meters = get_eval_res(model, eval_loader, opt, criterion)
    metrics, latest_file_paths = eval_epoch_post_processing(
        submission, opt, eval_dataset.data, save_submission_filename
    )
    return metrics, eval_loss_meters, latest_file_paths

In [54]:
def train_epoch(model, criterion, train_loader, optimizer, opt, epoch_i, **kwargs):
    """Trains for one epoch.

    Args:
        model: Model.
        criterion: Criterion.
        train_loader: Train loader.
        optimizer: Optimizer.
        opt: Options.
        epoch_i (int): Epoch index.
    """
    logger.info(f"[Epoch {epoch_i + 1}]")
    model.train()
    criterion.train()

    # init meters
    loss_meters = defaultdict(AverageMeter)

    num_training_examples = len(train_loader)
    timer_dataloading = time.time()
    for batch_idx, batch in tqdm(
        enumerate(train_loader), desc="Training Iteration", total=num_training_examples
    ):
        print("BATCH")
        print(batch)
        print("-----------------")
        model_inputs, targets = prepare_batch_inputs(batch[1], opt.device)

        outputs = (
            model(**model_inputs, targets=targets)
            if opt.model_name == "cg_detr"
            else model(**model_inputs)
        )
        loss_dict = criterion(outputs, targets)
        losses = sum(
            loss_dict[k] * criterion.weight_dict[k]
            for k in loss_dict.keys()
            if k in criterion.weight_dict
        )

        optimizer.zero_grad()
        losses.backward()

        if opt.grad_clip > 0:
            nn.utils.clip_grad_norm_(model.parameters(), opt.grad_clip)
        optimizer.step()

        loss_dict["loss_overall"] = float(losses)
        for k, v in loss_dict.items():
            loss_meters[k].update(
                float(v) * criterion.weight_dict[k]
                if k in criterion.weight_dict
                else float(v)
            )

    write_log(opt, epoch_i, loss_meters, kwargs)


from tqdm import trange


def train(
    model, criterion, optimizer, lr_scheduler, train_dataset, val_dataset, opt, **kwargs
):
    """Trains the model.

    Args:
        model: Model.
        criterion: Criterion.
        optimizer: Optimizer.
        lr_scheduler: LR scheduler.
        train_dataset: Train dataset.
        val_dataset: Val dataset.
        opt: Options.
    """
    opt.train_log_txt_formatter = "{time_str} [Epoch] {epoch:03d} [Loss] {loss_str}\n"
    opt.eval_log_txt_formatter = "{time_str} [Epoch] {epoch:03d} [Loss] {loss_str} [Metrics] {eval_metrics_str}\n"
    save_submission_filename = "latest_{}_val_preds.jsonl".format(opt.dset_name)

    train_loader = DataLoader(
        train_dataset,
        collate_fn=start_end_collate,
        batch_size=opt.bsz,
        num_workers=opt.num_workers,
        shuffle=True,
    )
    if opt.model_ema:
        logger.info("Using model EMA...")
        model_ema = ModelEMA(model, decay=opt.ema_decay)

    prev_best_score = 0
    for epoch_i in trange(opt.n_epoch, desc="Epoch"):
        train_epoch(model, criterion, train_loader, optimizer, opt, epoch_i)
        lr_scheduler.step()

        if opt.model_ema:
            model_ema.update(model)

        if (epoch_i + 1) % opt.eval_epoch_interval == 0:
            with torch.no_grad():
                if opt.model_ema:
                    metrics, eval_loss_meters, latest_file_paths = eval_epoch(
                        model_ema.module,
                        val_dataset,
                        opt,
                        save_submission_filename,
                        criterion,
                    )
                else:
                    metrics, eval_loss_meters, latest_file_paths = eval_epoch(
                        model, val_dataset, opt, save_submission_filename, criterion
                    )

            write_log(
                opt,
                epoch_i,
                eval_loss_meters,
                metrics=metrics,
                mode="val",
                kwargs=kwargs,
            )
            logger.info("metrics {}".format(pprint.pformat(metrics["brief"], indent=4)))

            stop_score = metrics["brief"]["MR-full-R1@0.7"]

            if stop_score > prev_best_score:
                prev_best_score = stop_score
                save_checkpoint(model, optimizer, lr_scheduler, epoch_i, opt)
                logger.info("The checkpoint file has been updated.")
                rename_latest_to_best(latest_file_paths)


def train_pipeline(opt, resume=None, is_wandb: bool = True):
    """Main function.

    Args:
        opt: Options.
        resume: Resume path.
    """
    logger.info("Setup config, data and model...")
    set_seed(opt.seed)

    # dataset & data loader
    dataset_config = EasyDict(
        data_path=opt.train_path,
        ctx_mode=opt.ctx_mode,
        a_feat_dir=opt.a_feat_dir,
        q_feat_dir=opt.t_feat_dir,
        q_feat_type="last_hidden_state",
        a_feat_type=opt.a_feat_type,
        max_q_l=opt.max_q_l,
        max_a_l=opt.max_a_l,
        clip_len=opt.clip_length,
        max_windows=opt.max_windows,
        span_loss_type=opt.span_loss_type,
        load_labels=True,
    )

    train_dataset = StartEndDataset(**dataset_config)
    copied_eval_config = copy.deepcopy(dataset_config)
    copied_eval_config.data_path = opt.eval_path
    eval_dataset = StartEndDataset(**copied_eval_config)
    kwargs = dict()
    if is_wandb:
        wandb_logger = WandbLogger(
            project_name="DCASE2026-Task6 TEST",
            run_name=get_run_name(
                prefix=train_opt.model_name,
                lr=train_opt.lr,
                batch_size=train_opt.bsz,
            ),
            config=train_opt,
        )
        kwargs["wandb_logger"] = wandb_logger
    # prepare model
    model, criterion, optimizer, lr_scheduler = setup_model(opt)

    logger.info(f"Model {model}")
    count_parameters(model, verbose=True)

    if resume is not None:
        checkpoint = torch.load(resume, weights_only=False)
        model.load_state_dict(checkpoint["model"])
        logger.info("Loaded model checkpoint: {}".format(resume))

    # logger.info("Start Training...")
    print("Start Training ...")

    # start training
    train(
        model,
        criterion,
        optimizer,
        lr_scheduler,
        train_dataset,
        eval_dataset,
        opt,
        **kwargs,
    )

#### Test Sample on Core Model

In [109]:
opt['device'] = 'mps'

In [110]:
# train_config_dir = "/content/preprocessed/train_config_clotho.yml"
train_config_dir = str(LOCAL_DIR / "config" / "config_pretraining.yml")



option_manager = BaseOptions(train_config_dir)
option_manager.parse()
train_opt = option_manager.option
train_opt.a_feat_dir = a_feat_dir
train_opt.t_feat_dir = q_feat_dir

In [111]:
train_opt['train_path'] = opt['train_path']

In [112]:
train_opt['train_path']

'/Users/minhld/workspace/projects/DCASE_2026/data/preprocessed/local_training_data.jsonl'

In [113]:
train_loader = DataLoader(
    train_dataset,
    collate_fn=start_end_collate,
    batch_size=train_opt.bsz,
    num_workers=train_opt.num_workers,
    shuffle=True,
)

In [115]:
# Create the iterator ONCE
data_iter = iter(train_loader)
sample = None

while sample is None:
    try:
        sample = next(data_iter)
    except StopIteration:
        # This prevents an infinite loop if the whole dataset is empty/broken
        print("Reached the end of the DataLoader without finding a valid sample.")
        break
    except Exception as e:
        # It is better to catch specific errors (like FileNotFoundError)
        print(f"Skipping broken sample: {e}")
        continue

In [174]:
model, criterion, optimizer, lr_scheduler = setup_model(train_opt)

logger.info(f"Model {model}")
count_parameters(model, verbose=True)

2026-05-09 13:35:37.508:INFO:__main__ - setup model/optimizer/scheduler
2026-05-09 13:35:37.545:INFO:__main__ - CUDA/MPS enabled.
2026-05-09 13:35:37.643:INFO:__main__ - Model QDDETR(
  (transformer): Transformer(
    (t2v_encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-1): 2 x T2V_TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
          )
          (linear1): Linear(in_features=256, out_features=1024, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=1024, out_features=256, bias=True)
          (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
          (activation): PReLU(num_parameters=1)
        )
     

Parameter Count: all 10,348,581; trainable 10,348,581; frozen 0


(10348581, 10348581)

In [ ]:
# # del model
# # clear_gpu_cache()

# import torch

# # 1. Define the shapes (matching your real data)
# batch_size = 32
# L_aud, D_aud = 61, 256
# L_txt, D_txt = 25, 256

# # 2. Create dummy tensors on the same device as the model
# # Using 'mps' if you are on Mac, otherwise 'cpu'
# device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# dummy_src_txt = torch.randn(batch_size, L_txt, D_txt).to(device)
# dummy_src_txt_mask = torch.ones(batch_size, L_txt).to(device)
# dummy_src_aud = torch.randn(batch_size, L_aud, D_aud).to(device)
# dummy_src_aud_mask = torch.ones(batch_size, L_aud).to(device)

# # 3. Wrap them in a tuple to be used as 'args'
# args = (model_inputs['src_txt'], model_inputs['src_txt_mask'], 
#         model_inputs['src_aud'], model_inputs['src_aud_mask'])

# # 4. Export
# torch.onnx.export(
#     model, 
#     args, 
#     "model.onnx",
#     input_names=["src_txt", "src_txt_mask", "src_aud", "src_aud_mask"],
#     output_names=["pred_logits", "pred_spans"],
#     opset_version=14 # Higher opset versions support more complex layers
# )

In [ ]:
"""
model_input: features and masks
targets: span_labels, saliency_pos_labels, saliency_neg_labels, saliency_all_labels
"""
model_inputs, targets = prepare_batch_inputs(sample[1], train_opt.device)

In [175]:
# print(f"Input device: {model_inputs.device}")
print(f"Model weight device: {next(model.parameters()).device}")

Model weight device: mps:0


In [176]:
outputs = (
    model(**model_inputs, targets=targets)
    if train_opt.model_name == "cg_detr"
    else model(**model_inputs)
)

In [177]:

loss_dict = criterion(outputs, targets)
losses = sum(
    loss_dict[k] * criterion.weight_dict[k]
    for k in loss_dict.keys()
    if k in criterion.weight_dict
)

In [178]:
losses

tensor(46.7961, device='mps:0', grad_fn=<AddBackward0>)

#### Training pipeline


In [ ]:
train_config_dir = "/content/preprocessed/train_config_clotho.yml"
option_manager = BaseOptions(train_config_dir)
option_manager.parse()
train_opt = option_manager.option
train_opt

In [ ]:
train_opt["train_path"] = str(
    Path("/content") / "preprocessed" / "clotho_moment_valid_train.jsonl"
)
train_opt["eval_path"] = str(
    Path("/content") / "preprocessed" / "clotho_moment_valid_val.jsonl"
)
train_opt["test_path"] = str(
    Path("/content") / "preprocessed" / "clotho_moment_valid_test.jsonl"
)
train_opt["a_feat_dir"] = str(Path("/content") / "clotho-moment" / "clap")
train_opt["t_feat_dir"] = str(Path("/content") / "clotho-moment" / "clap_text")

In [ ]:
train_pipeline(train_opt, is_wandb=True)

In [ ]:
"qid12669.npz" in os.listdir("/content/clotho-moment/clap_text")

In [ ]:
len(os.listdir("/content/clotho-moment/clap_text"))

In [ ]:
len(os.listdir("/content/clotho-moment/clap"))

In [ ]:
import os
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# 1. Pre-fetch file lists into sets for O(1) lookup speed
# This is often faster than the multithreading itself!
clap_text_files = set(os.listdir("/content/clotho-moment/clap_text"))
clap_files = set(os.listdir("/content/clotho-moment/clap"))

train_failed = []
train_jsonl = load_jsonl(
    filename="/content/preprocessed/clotho_moment_train_release.jsonl"
)


def check_files(data):
    """Worker function to check if files exist for a given entry."""
    qid_file = f"{data['qid']}.npz"
    vid_file = f"{data['vid']}.npz"

    if qid_file not in clap_text_files or vid_file not in clap_files:
        return data
    return None


# 2. Execute using ThreadPoolExecutor
# Max_workers=None defaults to a sensible number based on your CPU cores
with ThreadPoolExecutor() as executor:
    # list(tqdm(...)) ensures the progress bar tracks the submission of tasks
    results = list(tqdm(executor.map(check_files, train_jsonl), total=len(train_jsonl)))

# 3. Filter out the None values (successful checks) to get the failures
train_failed = [r for r in results if r is not None]

print(f"Total failed: {len(train_failed)}")

In [ ]:
"Bangkok_4209.0_4269.0.npz" in os.listdir("/content/clotho-moment/clap")

### Evaluating